# GUI-Model: Mobile UI Transition & Action Predictor

12 models × 2 training datasets (AndroidControl / MonkeyCollection) + MobiBench (eval-only benchmark).

**Stage 1 mode (full / lora)** 은 쉘 스크립트의 `--stage1-mode` 플래그로 선택한다 (default: `full`).
두 모드의 YAML 은 이 노트북에서 모두 생성되며, 산출물 경로·HF Hub ID 는 모드별로 분리된다.


## Dataset Overview

본 프로젝트는 세 가지 모바일 UI 인터랙션 데이터셋을 처리한다. **AndroidControl (AC)** 과 **MonkeyCollection (MC)** 은 Stage 1/2 학습 및 평가에 사용되고, **MobiBench (MB)** 는 평가 전용 벤치마크로만 사용된다.

### AndroidControl (AC)

AndroidControl 대규모 모바일 제어 데이터셋. Stage 1/2 양쪽 모두 학습·평가에 사용되며, 본 실험의 주력 데이터셋이다.

| 항목 | 값 |
|------|-----|
| Stage 1 (World Modeling) | 71,047건 |
| Stage 2 (Action Prediction) | 91,677건 |
| Train/Test Split | 95:5 (seed=42), Stage 2 는 app-level ID/OOD split |

**데이터 형식:** ShareGPT Multimodal 포맷.

**Action Type 분포 (Stage 2):**

| Type | 비율 |
|------|------|
| click | ~55.7% |
| finish | ~16.4% |
| scroll | ~11.9% |
| input | ~6.6% |
| open_app | ~6.1% |
| navigate_back | ~3.3% |
| long_click | ~0.2% |

### MonkeyCollection (MC)

Stage 1 (world modeling) 전용 대규모 데이터셋 (약 100K). AC 와 병행하여 Stage 1 학습/평가에 사용된다. Stage 2 데이터는 존재하지 않으므로 Stage 2 파이프라인에서는 자동 skip 된다.

| 항목 | 값 |
|------|-----|
| Stage 1 (World Modeling) | 약 100,000건 |
| Train/Test Split | 95:5 (seed=42) — `split_data.py --dataset MC` |

**데이터 형식:** AC 와 동일한 ShareGPT Multimodal 포맷.

### MobiBench (MB) — 평가 전용 벤치마크

MobiBench 는 더 이상 학습 데이터로 사용하지 않는다. AC/MC 로 학습한 모델의 일반화 성능을 평가하는 **out-of-distribution 벤치마크**로만 사용된다. `data/MobiBench/` 아래에는 `gui-model_stage1.jsonl` · `gui-model_stage2.jsonl` 두 단일 파일만 존재하며 (ID/OOD split 없음), `_action_eval.py` 가 single-pair 모드로 overall 1-섹션만 집계한다.

교차 평가 실행 (예):
```bash
bash scripts/stage1_eval.sh --model qwen3-vl-8b --train-dataset AC --eval-datasets AC,MC,MB
bash scripts/stage2_eval.sh --model qwen3-vl-8b --train-dataset AC --eval-datasets AC,MB \
     --stage1-mode full --stage1-epoch 3 --stage2-mode lora
```

> **학습 레시피**: AC 는 3 epochs, MC 는 3 epochs (baseline 동일). Stage 1 lora/full 양축 YAML 이 Cell 9/11 에서 자동 생성된다.
> **AC 하이퍼파라미터는 모델 크기 3 단(2B / 3-4B / 7-9B) 으로 공유**되며, `_SIZE_CONFIG_AC` (Cell 6) 에서 관리한다. MC 는 tier 미적용 — dataset baseline + per-model override 구조.
> Code2World, gWorld, MobileDreamer 논문 + vlm-gui-finetuning-research.md 를 참고하여 설계되었다.

## 0. Environment Setup

프로젝트 경로 설정, GPU 확인, 의존성 설치를 수행합니다.

> **`BASE_DIR`을 본인의 프로젝트 경로로 수정하세요.**

In [ ]:
%%bash
set -e
# 0. 서브프로젝트 clone (없을 때만).
if [ ! -d LlamaFactory ]; then
  git clone https://github.com/hiyouga/LlamaFactory.git
fi

# 1. 서브프로젝트 editable 설치.
#    LF transitive 상한 (transformers<=5.2.0) 과 우리 extras (transformers>=4.56.0,<5)
#    가 4.56–4.57.x 구간에서 겹치므로 --no-deps 우회는 더 이상 필요 없다.
PIP_USER=0 pip install --no-user -e ./LlamaFactory

# 2. 루트 extras — LlamaFactory 런타임/평가 deps 재선언 포함
#    (transformers>=4.56.0,<5, vllm>=0.11.2, av, fire, datasets 등). setup.py::LLAMAFACTORY 참고.
PIP_USER=0 pip install --no-user -e '.[llamafactory]'

### AC 크기-tier 하이퍼파라미터 (`_SIZE_CONFIG_AC`)

AC(AndroidControl) 은 모델 크기 3 단 (**2B / 3-4B / 7-9B**) 의 공유 하이퍼파라미터로 운영합니다. 아래 tier 값 위에 모델별 delta 만 `hparam_overrides` 로 얹힙니다. MC(MonkeyCollection) 는 tier 미적용 — dataset baseline + per-model override 구조 그대로. MB 는 평가 전용 벤치마크이므로 학습 하이퍼파라미터 해석에서 제외됩니다.

**크기 배정 (총 8 모델)**:

| 구간 | 모델 |
|---|---|
| **2B** | qwen2-vl-2b |
| **3-4B** | qwen2.5-vl-3b · qwen3-vl-4b · qwen3.5-4b-base |
| **7-9B** | qwen2-vl-7b · qwen2.5-vl-7b · qwen3-vl-8b · qwen3.5-9b-base |

**Stage 1 (full FT)**:

| 구간 | lr | warmup_ratio | max_grad_norm |
|---|---|---|---|
| 2B | 1.5e-5 | 0.08 | 0.5 |
| 3-4B | 1.2e-5 | 0.06 | 0.5 |
| 7-9B | 1.0e-5 (baseline) | 0.03 (baseline) | 1.0 (baseline) |

**Stage 1 LoRA** (`stage1_full` 위에 덮어쓰기):

| 구간 | lr | LoRA r / α | dropout |
|---|---|---|---|
| 2B | 1.5e-4 | 8 / 16 | 0.05 |
| 3-4B | 1.2e-4 | 12 / 24 | 0.05 |
| 7-9B | 1.0e-4 | 16 / 32 | 0.05 |

**Stage 2 (LoRA)**:

| 구간 | lr | LoRA r / α | dropout | warmup | epochs |
|---|---|---|---|---|---|
| 2B | 6.0e-5 | 16 / 32 | 0.05 | 0.05 | 3 |
| 3-4B | 5.0e-5 | 24 / 48 | 0.05 | 0.04 | 3 |
| 7-9B | 4.0e-5 | 32 / 64 | 0.05 | 0.03 | 3 |

> **설계 근거**: `outputs/AC/eval/qwen{2.5-vl-7b,3-vl-8b}/stage2_eval` 실측에서 ① lr 5e-5 가 7-9B 에서 상단 경계 (7B e3 retrograde, 8B cond_text 퇴화), ② dropout 0.10 이 저빈도 action type (input 6.6%, nav_back 3.3%) 을 불안정하게 만듦이 확인됨. 2B / 3-4B 는 Stage 1 크기 규칙을 Stage 2 에 이식한 외삽.

> **참고 우선순위** (merge 순서): `_DATASET_CONFIG` baseline → `_SIZE_CONFIG_AC[size]` → `_MODEL_CONFIG[model].hparam_overrides`. 아래 Cell 6 의 CONFIGS 빌더가 이 순서로 `.update()` 합니다.


In [ ]:
import os
from dotenv import load_dotenv

# ============================================================
# === Project root path ===
BASE_DIR = "./"
# ============================================================

BASE_DIR = os.path.abspath(BASE_DIR)
LF_ROOT = os.path.join(BASE_DIR, "LlamaFactory")

load_dotenv(os.path.join(BASE_DIR, ".env"))

# ============================================================
# === Global batch size & NPROC-aware grad_accum ===
# global batch 가 NPROC_PER_NODE 변경에 흔들리지 않도록
# gradient_accumulation_steps 를 자동 역계산한다.
#
#   global_batch = per_device_train_batch_size * grad_accum * NPROC_PER_NODE
#   grad_accum   = GLOBAL_BATCH_SIZE / (per_device * NPROC_PER_NODE)
# ============================================================
NPROC_PER_NODE = int(os.getenv("NPROC_PER_NODE", "2"))
GPU_TYPE = os.getenv("GPU_TYPE", "H100").upper()
GLOBAL_BATCH_SIZE = 64

_ALLOWED_GPU_TYPES = ("RTX5090", "A100", "H100")
_ALLOWED_NPROC = (1, 2, 4, 8)
if GPU_TYPE not in _ALLOWED_GPU_TYPES:
    raise ValueError(
        f"Unsupported GPU_TYPE={GPU_TYPE!r}. "
        f"Allowed: {_ALLOWED_GPU_TYPES}. Set in .env."
    )
if NPROC_PER_NODE not in _ALLOWED_NPROC:
    raise ValueError(
        f"Unsupported NPROC_PER_NODE={NPROC_PER_NODE}. "
        f"Allowed: {_ALLOWED_NPROC}. Set in .env."
    )

# === per_device_train_batch_size by (model size, GPU type) ============
# RTX5090=32GB, A100=80GB, H100=80GB. 8개 모델 모두 LlamaFactory backend.
# 값을 바꾸면 GLOBAL_BATCH_SIZE / (per_device * NPROC_PER_NODE) 가
# 정수가 되도록 유지해야 한다 (_derive_grad_accum 가 강제 검증).
_PER_DEVICE_BS_BY_SIZE = {
    "2B":   {"RTX5090": 4, "A100": 8, "H100": 8},
    "3-4B": {"RTX5090": 2, "A100": 4, "H100": 4},
    "7-9B": {"RTX5090": 1, "A100": 2, "H100": 2},
}


def lf_per_device_bs(size: str) -> int:
    """Return per_device_train_batch_size for given model size on current GPU."""
    return _PER_DEVICE_BS_BY_SIZE[size][GPU_TYPE]


def _derive_grad_accum(per_device: int, nproc: int = NPROC_PER_NODE,
                       target: int = GLOBAL_BATCH_SIZE) -> int:
    """Return gradient_accumulation_steps so that per_device * ga * nproc == target."""
    denom = per_device * nproc
    if denom <= 0 or target % denom != 0:
        raise ValueError(
            f"GLOBAL_BATCH_SIZE({target}) is not divisible by "
            f"per_device({per_device}) * NPROC_PER_NODE({nproc}) = {denom}. "
            f"Adjust NPROC_PER_NODE / GPU_TYPE in .env or GLOBAL_BATCH_SIZE."
        )
    return target // denom

# ============================================================
# === Model-family image-pixel configs ===
# Qwen 계열별 vision encoder patch-size (factor) 와 token budget.
# 각 family 의 max/min pixels 는 학습 YAML 의 image_max_pixels /
# image_min_pixels 필드에 그대로 주입된다. factor / merged_tokens /
# vertical_retention 은 데이터 전처리 메타이며 YAML 에는 들어가지 않는다.
#
# Token budget 은 학습 데이터셋에 따라 결정된다 (학습-추론 mismatch 방지).
#   - family default                 : max_tokens=2048   (AC, MC 학습 + 평가)
#   - AC2 (AndroidControl_2) override: max_tokens=5400   (AC2 학습 + 해당 모델의
#     모든 평가에 동일 budget 사용 — _common.sh::build_infer_cmd 가 TRAIN_DATASET
#     로 분기). family 별 factor 에 따라 max_pixels = max_tokens × factor² 로 환산.
#   Qwen2/2.5-VL  (factor 28): 2048→1,605,632, 5400→4,233,600
#   Qwen3-VL/3.5  (factor 32): 2048→2,097,152, 5400→5,529,600
# ============================================================
QWEN2_VL_CONFIG = {
    "max_pixels": 1_605_632,      # 2048 x 28²
    "min_pixels": 3_136,          # 4 x 28²
    "factor": 28,
    "max_tokens": 2048,
    "merged_tokens_at_1080x2400": 510,
    "vertical_retention": 0.782,
}
QWEN2_5_VL_CONFIG = {  # Qwen2-VL 과 동일
    "max_pixels": 1_605_632, "min_pixels": 3_136, "factor": 28, "max_tokens": 2048,
    "merged_tokens_at_1080x2400": 510, "vertical_retention": 0.782,
}
QWEN3_VL_CONFIG = {
    "max_pixels": 2_097_152,      # 2048 x 32²
    "min_pixels": 4_096,          # 4 x 32²
    "factor": 32,
    "max_tokens": 2048,
    "merged_tokens_at_1080x2400": 510,
    "vertical_retention": 0.893,
}
QWEN3_5_VL_CONFIG = {  # Qwen3-VL 과 동일 (사용자 지시)
    "max_pixels": 2_097_152, "min_pixels": 4_096, "factor": 32, "max_tokens": 2048,
    "merged_tokens_at_1080x2400": 510, "vertical_retention": 0.893,
}

MODEL_FAMILY_CONFIG = {
    "qwen2-vl-2b":     QWEN2_VL_CONFIG,
    "qwen2-vl-7b":     QWEN2_VL_CONFIG,
    "qwen2.5-vl-3b":   QWEN2_5_VL_CONFIG,
    "qwen2.5-vl-7b":   QWEN2_5_VL_CONFIG,
    "qwen3-vl-4b":     QWEN3_VL_CONFIG,
    "qwen3-vl-8b":     QWEN3_VL_CONFIG,
    "qwen3.5-4b-base": QWEN3_5_VL_CONFIG,
    "qwen3.5-9b-base": QWEN3_5_VL_CONFIG,
}

# ============================================================
# === Model Registry (8 models) ===
# AC 전용으로 모델 크기 3 단 tier (2B / 3-4B / 7-9B) 의 공유 하이퍼파라미터
# (_SIZE_CONFIG_AC) 를 적용한다. MonkeyCollection 은 tier 미적용.
#
# 적용 우선순위 (merge 순서):
#   1. _DATASET_CONFIG[ds].stage{1,2}              (dataset baseline)
#   2. _SIZE_CONFIG_AC[size].stage{1, 1_lora, 2}   (AC 일 때만)
#   3. _MODEL_CONFIG[model].hparam_overrides       (계열 delta)
#
# image_max_pixels / image_min_pixels 는 MODEL_FAMILY_CONFIG 에서 자동 주입.
# cutoff_len: stage1=8192, stage2=10000.
# ============================================================
def _img_cfg(short):
    f = MODEL_FAMILY_CONFIG[short]
    return {"image_max_pixels": f["max_pixels"], "image_min_pixels": f["min_pixels"]}

_MODEL_CONFIG = {
    "qwen2-vl-2b":   {"model_id": "Qwen/Qwen2-VL-2B-Instruct",   "short_name": "qwen2-vl-2b",   "template": "qwen2_vl",         "size": "2B",
        **_img_cfg("qwen2-vl-2b"),   "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4, "hparam_overrides": {}},
    "qwen2-vl-7b":   {"model_id": "Qwen/Qwen2-VL-7B-Instruct",   "short_name": "qwen2-vl-7b",   "template": "qwen2_vl",         "size": "7-9B",
        **_img_cfg("qwen2-vl-7b"),   "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4, "hparam_overrides": {}},
    "qwen2.5-vl-3b": {"model_id": "Qwen/Qwen2.5-VL-3B-Instruct", "short_name": "qwen2.5-vl-3b", "template": "qwen2_vl",         "size": "3-4B",
        **_img_cfg("qwen2.5-vl-3b"), "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4, "hparam_overrides": {}},
    "qwen2.5-vl-7b": {"model_id": "Qwen/Qwen2.5-VL-7B-Instruct", "short_name": "qwen2.5-vl-7b", "template": "qwen2_vl",         "size": "7-9B",
        **_img_cfg("qwen2.5-vl-7b"), "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4, "hparam_overrides": {}},
    "qwen3-vl-4b":   {"model_id": "Qwen/Qwen3-VL-4B-Instruct",   "short_name": "qwen3-vl-4b",   "template": "qwen3_vl_nothink", "size": "3-4B",
        **_img_cfg("qwen3-vl-4b"),   "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4, "hparam_overrides": {}},
    "qwen3-vl-8b":   {"model_id": "Qwen/Qwen3-VL-8B-Instruct",   "short_name": "qwen3-vl-8b",   "template": "qwen3_vl_nothink", "size": "7-9B",
        **_img_cfg("qwen3-vl-8b"),   "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4, "hparam_overrides": {}},
    "qwen3.5-4b-base": {"model_id": "Qwen/Qwen3.5-4B-Base", "short_name": "qwen3.5-4b-base", "template": "qwen3_5_nothink", "size": "3-4B",
        **_img_cfg("qwen3.5-4b-base"), "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4, "hparam_overrides": {}},
    "qwen3.5-9b-base": {"model_id": "Qwen/Qwen3.5-9B-Base", "short_name": "qwen3.5-9b-base", "template": "qwen3_5_nothink", "size": "7-9B",
        **_img_cfg("qwen3.5-9b-base"), "freeze_vision_tower": True, "stage1_deepspeed": "examples/deepspeed/ds_z3_config.json", "stage1_nproc": 4, "hparam_overrides": {}},
}

# ============================================================
# === Dataset configs (baseline hparams per dataset) ===
# 학습 대상 DS 는 {AC, MC} 만. MB 는 평가 전용.
# ============================================================
_DATASET_CONFIG = {
    "AndroidControl": {
        "lf_subfolder": "GUI-Model-AC",
        "ds_prefix": "GUI-Model-AC",
        "output_prefix": "AC/",
        "hf_slug": "ac-",
        "stage1": {
            "lr": "1.0e-5", "epochs": 3, "warmup_ratio": 0.03,
            "save_strategy": "epoch", "save_steps": None,
            "eval_strategy": "epoch", "eval_steps": None,
            "per_device_eval_batch_size": 4,
            "lora_rank": 8, "lora_alpha": 16, "lora_dropout": 0.05,
            "weight_decay": 0.01, "max_grad_norm": 1.0,
            "lr_scheduler_type": "cosine",
        },
        "stage2": {
            "lr": "5.0e-5", "epochs": 3, "warmup_ratio": 0.03,
            "save_strategy": "epoch", "save_steps": None,
            "eval_strategy": "epoch", "eval_steps": None,
            "per_device_eval_batch_size": 4,
            "lora_rank": 32, "lora_alpha": 64, "lora_dropout": 0.1,
            "weight_decay": 0.01, "max_grad_norm": 1.0,
            "lr_scheduler_type": "cosine",
        },
    },
    "AndroidControl_2": {
        "lf_subfolder": "GUI-Model-AC_2",
        "ds_prefix": "GUI-Model-AC_2",
        "output_prefix": "AC_2/",
        "hf_slug": "ac-2-",
        "stage1": {
            "lr": "1.0e-5", "epochs": 3, "warmup_ratio": 0.03,
            "save_strategy": "epoch", "save_steps": None,
            "eval_strategy": "epoch", "eval_steps": None,
            "per_device_eval_batch_size": 4,
            "lora_rank": 8, "lora_alpha": 16, "lora_dropout": 0.05,
            "weight_decay": 0.01, "max_grad_norm": 1.0,
            "lr_scheduler_type": "cosine",
        },
        "stage2": {
            "lr": "5.0e-5", "epochs": 3, "warmup_ratio": 0.03,
            "save_strategy": "epoch", "save_steps": None,
            "eval_strategy": "epoch", "eval_steps": None,
            "per_device_eval_batch_size": 4,
            "lora_rank": 32, "lora_alpha": 64, "lora_dropout": 0.1,
            "weight_decay": 0.01, "max_grad_norm": 1.0,
            "lr_scheduler_type": "cosine",
        },
        # AC2 학습 시 image budget 을 5400 tokens 로 상향. family 별 factor 에
        # 따라 max_pixels 가 환산된다 (Qwen2/2.5-VL=4,233,600, Qwen3-VL/3.5=5,529,600).
        # min_tokens 는 family default(=4) 유지. 평가측 _common.sh::build_infer_cmd
        # 가 TRAIN_DATASET=AC_2 일 때 동일 budget 으로 추론한다.
        "image_overrides": {"max_tokens": 5400},
    },
    "MonkeyCollection": {
        "lf_subfolder": "GUI-Model-MC",
        "ds_prefix": "GUI-Model-MC",
        "output_prefix": "MC/",
        "hf_slug": "mc-",
        "stage1": {
            "lr": "1.0e-5", "epochs": 3, "warmup_ratio": 0.03,
            "save_strategy": "epoch", "save_steps": None,
            "eval_strategy": "epoch", "eval_steps": None,
            "per_device_eval_batch_size": 4,
            "lora_rank": 8, "lora_alpha": 16, "lora_dropout": 0.05,
            "weight_decay": 0.01, "max_grad_norm": 1.0,
            "lr_scheduler_type": "cosine",
        },
        # Stage 2 MC 미지원 — placeholder. `_STAGE1_ONLY` guard 로 skip.
        "stage2": {
            "lr": "5.0e-5", "epochs": 3, "warmup_ratio": 0.03,
            "save_strategy": "epoch", "save_steps": None,
            "eval_strategy": "epoch", "eval_steps": None,
            "per_device_eval_batch_size": 4,
            "lora_rank": 32, "lora_alpha": 64, "lora_dropout": 0.1,
            "weight_decay": 0.01, "max_grad_norm": 1.0,
            "lr_scheduler_type": "cosine",
        },
    },
}

_STAGE1_ONLY = {"MonkeyCollection"}

# ID/OOD split 없이 `gui-model_stage{1,2}_test.jsonl` 단일 파일을 사용하는 데이터셋.
# `_STAGE1_ONLY` 와 직교 — MC 는 Stage 1 만 + 단일 test, AC2 는 Stage 1+2 모두 단일 test.
_SINGLE_TEST = {"MonkeyCollection", "AndroidControl_2"}

_EVAL_ONLY_BENCHMARKS = {
    "MobiBench": {
        "ds_prefix": "GUI-Model-MB",
        "data_dir": os.path.join(BASE_DIR, "data", "MobiBench"),
        "stage1_jsonl": "gui-model_stage1.jsonl",
        "stage2_jsonl": "gui-model_stage2.jsonl",
        "ds_s1_name": "GUI-Model-MB_stage1",
        "ds_s2_name": "GUI-Model-MB_stage2",
    },
}

# ============================================================
# === Size-tier shared hparams (AC only) ===
# Stage 1 (full FT):
#   2B   : lr 1.5e-5, warmup 0.08, max_grad_norm 0.5
#   3-4B : lr 1.2e-5, warmup 0.06, max_grad_norm 0.5
#   7-9B : dataset baseline 유지
# Stage 1 LoRA (stage1_full 위에 추가 덮어쓰기):
#   2B   : lr 1.5e-4, r=8/a=16
#   3-4B : lr 1.2e-4, r=12/a=24
#   7-9B : lr 1.0e-4, r=16/a=32
# Stage 2 (LoRA):
#   2B   : lr 6.0e-5, warmup 0.05, r=16/a=32
#   3-4B : lr 5.0e-5, warmup 0.04, r=24/a=48
#   7-9B : lr 4.0e-5, r=32/a=64
# ============================================================
_SIZE_CONFIG_AC = {
    "2B": {
        "stage1":      {"lr": "1.5e-5", "warmup_ratio": 0.08, "max_grad_norm": 0.5},
        "stage1_lora": {"lr": "1.5e-4", "lora_rank": 8,  "lora_alpha": 16, "lora_dropout": 0.05},
        "stage2":      {"lr": "6.0e-5", "warmup_ratio": 0.05,
                        "lora_rank": 16, "lora_alpha": 32, "lora_dropout": 0.05},
    },
    "3-4B": {
        "stage1":      {"lr": "1.2e-5", "warmup_ratio": 0.06, "max_grad_norm": 0.5},
        "stage1_lora": {"lr": "1.2e-4", "lora_rank": 12, "lora_alpha": 24, "lora_dropout": 0.05},
        "stage2":      {"lr": "5.0e-5", "warmup_ratio": 0.04,
                        "lora_rank": 24, "lora_alpha": 48, "lora_dropout": 0.05},
    },
    "7-9B": {
        "stage1":      {},
        "stage1_lora": {"lr": "1.0e-4", "lora_rank": 16, "lora_alpha": 32, "lora_dropout": 0.05},
        "stage2":      {"lr": "4.0e-5", "lora_dropout": 0.05},
    },
}

# ============================================================
# === Model ordering for execution cells ===
# ============================================================
MODEL_ORDER = [
    ("qwen2-vl-2b",     "Qwen2-VL-2B"),
    ("qwen2-vl-7b",     "Qwen2-VL-7B"),
    ("qwen2.5-vl-3b",   "Qwen2.5-VL-3B"),
    ("qwen2.5-vl-7b",   "Qwen2.5-VL-7B"),
    ("qwen3-vl-4b",     "Qwen3-VL-4B"),
    ("qwen3-vl-8b",     "Qwen3-VL-8B"),
    ("qwen3.5-4b-base", "Qwen3.5-4B-Base"),
    ("qwen3.5-9b-base", "Qwen3.5-9B-Base"),
]
DS_ORDER = [("AC", "AndroidControl"), ("AC_2", "AndroidControl_2"), ("MC", "MonkeyCollection")]

# ============================================================
# === Build CONFIGS: CONFIGS[model_key][ds_name] ===
# ============================================================
CONFIGS = {}
for _model_key, _mcfg in _MODEL_CONFIG.items():
    CONFIGS[_model_key] = {}
    _overrides = _mcfg.get("hparam_overrides", {})
    _size = _mcfg["size"]
    _per_device = lf_per_device_bs(_size)
    _grad_accum = _derive_grad_accum(_per_device)
    for _ds_name, _cfg in _DATASET_CONFIG.items():
        c = dict(_cfg)
        # image budget: family default 위에 dataset image_overrides 를 덮어쓴다.
        # override 키는 token 단위 ("max_tokens", "min_tokens") 또는 px 단위
        # ("image_max_pixels", "image_min_pixels"). token 키는 family factor² 로 환산.
        _factor = MODEL_FAMILY_CONFIG[_model_key]["factor"]
        _img = dict(_img_cfg(_model_key))
        for _ok, _ov in _cfg.get("image_overrides", {}).items():
            if _ok == "max_tokens":
                _img["image_max_pixels"] = _ov * _factor * _factor
            elif _ok == "min_tokens":
                _img["image_min_pixels"] = _ov * _factor * _factor
            else:
                _img[_ok] = _ov
        c["image_max_pixels"] = _img["image_max_pixels"]
        c["image_min_pixels"] = _img["image_min_pixels"]
        c["model_key"] = _model_key
        c["model_id"] = _mcfg["model_id"]
        c["short_name"] = _mcfg["short_name"]
        c["template"] = _mcfg["template"]
        c["model_config"] = _mcfg
        c["dataset_name"] = _ds_name
        c["data_dir"] = os.path.join(BASE_DIR, "data", _ds_name)
        c["ds_s1_train"] = f"{c['ds_prefix']}_stage1_train"
        if _ds_name in _SINGLE_TEST:
            c["ds_s1_test"] = f"{c['ds_prefix']}_stage1_test"
        else:
            c["ds_s1_test_id"]  = f"{c['ds_prefix']}_stage1_test_id"
            c["ds_s1_test_ood"] = f"{c['ds_prefix']}_stage1_test_ood"
            c["ds_s1_test"]     = c["ds_s1_test_id"]
        c["ds_s2_train"] = f"{c['ds_prefix']}_stage2_train"
        if _ds_name in _SINGLE_TEST:
            c["ds_s2_test"] = f"{c['ds_prefix']}_stage2_test"
        else:
            c["ds_s2_test_id"]  = f"{c['ds_prefix']}_stage2_test_id"
            c["ds_s2_test_ood"] = f"{c['ds_prefix']}_stage2_test_ood"
            c["ds_s2_test"]     = c["ds_s2_test_id"]

        c["hf_s1_model_full"] = f"SaFD-00/{_mcfg['short_name']}-{c['hf_slug']}stage1-full-world-model"
        c["hf_s1_model_lora"] = f"SaFD-00/{_mcfg['short_name']}-{c['hf_slug']}stage1-lora-world-model"
        c["hf_s1_model"] = c["hf_s1_model_full"]

        c["hf_s2_base"] = f"SaFD-00/{_mcfg['short_name']}-{c['hf_slug']}stage2-base"
        c["hf_s2_world_full"] = f"SaFD-00/{_mcfg['short_name']}-{c['hf_slug']}stage2-full-world-model"
        c["hf_s2_world_lora"] = f"SaFD-00/{_mcfg['short_name']}-{c['hf_slug']}stage2-lora-world-model"
        c["hf_s2_world"] = c["hf_s2_world_full"]

        _ds_code = c["output_prefix"].rstrip("/")
        _mshort  = _mcfg["short_name"]

        c["save_s1_full"]        = f"../outputs/{_ds_code}/adapters/{_mshort}_stage1_full"
        c["save_s1_lora"]        = f"../outputs/{_ds_code}/adapters/{_mshort}_stage1_lora"
        c["out_s1_merged_full"]  = f"../outputs/{_ds_code}/merged/{_mshort}_stage1_full"
        c["out_s1_merged_lora"]  = f"../outputs/{_ds_code}/merged/{_mshort}_stage1_lora"
        c["save_s1"]        = c["save_s1_full"]
        c["out_s1_merged"]  = c["out_s1_merged_full"]

        for _m2 in ("full", "lora"):
            c[f"save_s2_{_m2}_base"]                 = f"../outputs/{_ds_code}/adapters/{_mshort}_stage2_{_m2}_base"
            c[f"save_s2_{_m2}_world_from_full"]      = f"../outputs/{_ds_code}/adapters/{_mshort}_stage2_{_m2}_world_model_from_full"
            c[f"save_s2_{_m2}_world_from_lora"]      = f"../outputs/{_ds_code}/adapters/{_mshort}_stage2_{_m2}_world_model_from_lora"
            c[f"out_s2_merged_{_m2}_base"]           = f"../outputs/{_ds_code}/merged/{_mshort}_stage2_{_m2}_base"
            c[f"out_s2_merged_{_m2}_world_from_full"]= f"../outputs/{_ds_code}/merged/{_mshort}_stage2_{_m2}_world_model_from_full"
            c[f"out_s2_merged_{_m2}_world_from_lora"]= f"../outputs/{_ds_code}/merged/{_mshort}_stage2_{_m2}_world_model_from_lora"
        c["save_s2_base"]                 = c["save_s2_lora_base"]
        c["save_s2_world_from_full"]      = c["save_s2_lora_world_from_full"]
        c["save_s2_world_from_lora"]      = c["save_s2_lora_world_from_lora"]
        c["out_s2_merged_base"]           = c["out_s2_merged_lora_base"]
        c["out_s2_merged_world_from_full"]= c["out_s2_merged_lora_world_from_full"]
        c["out_s2_merged_world_from_lora"]= c["out_s2_merged_lora_world_from_lora"]
        c["save_s2_world"]                = c["save_s2_world_from_full"]
        c["out_s2_merged_world"]          = c["out_s2_merged_world_from_full"]

        _tier = _SIZE_CONFIG_AC[_size] if _ds_name in {"AndroidControl", "AndroidControl_2"} else None

        s1_full = dict(c["stage1"])
        if _tier is not None:
            s1_full.update(_tier.get("stage1", {}))
        s1_full.update(_overrides.get("stage1", {}))

        s1_lora = dict(s1_full)
        if _tier is not None:
            s1_lora.update(_tier.get("stage1_lora", {}))
        s1_lora.update(_overrides.get("stage1_lora", {}))

        s2 = dict(c["stage2"])
        if _tier is not None:
            s2.update(_tier.get("stage2", {}))
        s2.update(_overrides.get("stage2", {}))

        s1_full["gradient_accumulation_steps"] = _grad_accum
        s1_lora["gradient_accumulation_steps"] = _grad_accum
        s2["gradient_accumulation_steps"] = _grad_accum

        c["stage1_full"] = s1_full
        c["stage1_lora"] = s1_lora
        c["stage1"]      = s1_full
        c["stage2"]      = s2
        c["stage1_only"] = _ds_name in _STAGE1_ONLY
        c["per_device_train_batch_size"] = _per_device

        CONFIGS[_model_key][_ds_name] = c

print(f"Working directory: {os.getcwd()}")
print(f"LlamaFactory root: {LF_ROOT}")
print(f"GPU_TYPE={GPU_TYPE}, NPROC_PER_NODE={NPROC_PER_NODE}, GLOBAL_BATCH_SIZE={GLOBAL_BATCH_SIZE}")
print("  per_device_train_batch_size × grad_accum  (target global={}):".format(GLOBAL_BATCH_SIZE))
for _sz in ("2B", "3-4B", "7-9B"):
    _pd = lf_per_device_bs(_sz)
    _ga = _derive_grad_accum(_pd)
    print(f"    {_sz:5s}: {_pd} × {_ga} × {NPROC_PER_NODE} = {_pd * _ga * NPROC_PER_NODE}")
print(f"Models: {list(CONFIGS.keys())}")
print(f"Train datasets: {list(_DATASET_CONFIG.keys())}")
print(f"Stage1-only datasets: {sorted(_STAGE1_ONLY)}")
print(f"Eval-only benchmarks: {list(_EVAL_ONLY_BENCHMARKS.keys())}")

# --- AC size-tier summary (per model, AC only) ---
print("\n=== AndroidControl size-tier resolution ===")
_size_order = {"2B": 0, "3-4B": 1, "7-9B": 2}
for _mk in sorted(CONFIGS.keys(), key=lambda k: (_size_order[_MODEL_CONFIG[k]["size"]], k)):
    c = CONFIGS[_mk]["AndroidControl"]
    s1f, s1l, s2 = c["stage1_full"], c["stage1_lora"], c["stage2"]
    fam = MODEL_FAMILY_CONFIG[_mk]
    print(f"  {_mk:18s} size={_MODEL_CONFIG[_mk]['size']:5s}  "
          f"img={fam['max_pixels']}/f{fam['factor']}  "
          f"s1 lr={s1f['lr']} wu={s1f['warmup_ratio']}  "
          f"s2 lr={s2['lr']} r={s2['lora_rank']}/a={s2['lora_alpha']}/d={s2['lora_dropout']}")


### YAML Configs 일괄 생성

이 블록은 `LlamaFactory/examples/custom/` 하위 학습 YAML 을 **환경설정 직후 한 번에 생성**합니다. 이후 학습 단계에서 별도의 YAML 생성 셀 실행 없이 즉시 스크립트/CLI 를 호출할 수 있습니다.


**포함 대상 (12개 모델 × 2개 데이터셋)**

| 블록 | 생성 파일 | 비고 |
|------|-----------|------|
| `[LlamaFactory]` Stage 1 Training YAML | `custom/GUI-Model-{MB,AC}/stage1_{full,lora}/{MODEL}_world-model.yaml` | full / lora 양쪽 생성 (Cell 9) |
| `[LlamaFactory]` Stage 2 Training YAML | `custom/GUI-Model-{MB,AC}/stage2_{full,lora}/{MODEL}_{base,world-model-full,world-model-lora}.yaml` | full / lora × 3 variant = 6 YAML (Cell 13) |


**HF repo id 규칙** (`_common.sh`):
- Stage 1 : `SaFD-00/{short}-{slug}world-model-stage1-{full|lora}-epoch{E}`
- Stage 2 base : `SaFD-00/{short}-{slug}base-stage2-{full|lora}-epoch{E2}`
- Stage 2 world-model : `SaFD-00/{short}-{slug}world-model-stage1-{M1}-epoch{E1}-stage2-{M2}-epoch{E2}`


#### [LlamaFactory] Stage 1 Training YAML

`LlamaFactory/examples/custom/GUI-Model-{MB,AC}/stage1_{full,lora}/{MODEL}_world-model.yaml` 생성. LF_ROOT 기준 상대경로 (`../outputs/`) 사용. Qwen 계열 8개 모델 전체에 대해 생성된다.

> **DeepSpeed config 분기**
> - `full` 모드: 모델별 `stage1_deepspeed` (기본 `examples/deepspeed/ds_z3_config.json`) 그대로 사용.
> - `lora` 모드:
>   - `GPU_TYPE=RTX5090` (32GB VRAM): `examples/deepspeed/ds_z3_offload_config.json` 로 swap — ZeRO-3 + CPU offload 로 단일 GPU OOM 회피.
>   - 그 외 (`A100` / `H100`, 80GB VRAM): `examples/deepspeed/ds_z3_config.json` (offload 없는 표준 ZeRO-3) 사용.

In [ ]:
from pathlib import Path

# LF Stage 1 YAML — full / lora 두 벌 생성.
# LoRA mode 는 finetuning_type: lora + lora_rank/alpha/target/dropout 블록을 추가.
# LoRA + GPU_TYPE=RTX5090 시 ZeRO-3 offload config 로 swap (32GB VRAM 1장 OOM 회피); 그 외는 mcfg.stage1_deepspeed 그대로.
for MODE in ["full", "lora"]:
    for model_key, ds_configs in CONFIGS.items():
        for ds_name, cfg in ds_configs.items():
            s1 = cfg[f"stage1_{MODE}"]
            mcfg = cfg["model_config"]
            yaml_dir = Path(LF_ROOT) / "examples" / "custom" / cfg["lf_subfolder"] / f"stage1_{MODE}"
            yaml_dir.mkdir(parents=True, exist_ok=True)

            save_steps_line = f"save_steps: {s1['save_steps']}\n" if s1["save_strategy"] == "steps" else ""
            _ds_path = mcfg.get("stage1_deepspeed")
            if _ds_path and MODE == "lora" and GPU_TYPE == "RTX5090":
                _ds_path = "examples/deepspeed/ds_z3_offload_config.json"
            ds_line = f"deepspeed: {_ds_path}\n" if _ds_path else ""
            optim_line = f"optim: {s1['optim']}\n" if s1.get("optim") else ""
            seed_line = f"seed: {s1['seed']}\n" if s1.get("seed") is not None else ""

            output_dir = cfg[f"save_s1_{MODE}"]

            if MODE == "full":
                method_block = f"""\
### method
stage: sft
do_train: true
finetuning_type: full
freeze_vision_tower: {str(mcfg["freeze_vision_tower"]).lower()}"""
            else:
                method_block = f"""\
### method
stage: sft
do_train: true
finetuning_type: lora
freeze_vision_tower: {str(mcfg["freeze_vision_tower"]).lower()}
lora_rank: {s1["lora_rank"]}
lora_alpha: {s1["lora_alpha"]}
lora_target: all
lora_dropout: {s1["lora_dropout"]}"""

            STAGE1_CONFIG = f"""\
### model
model_name_or_path: {cfg["model_id"]}
trust_remote_code: true
image_max_pixels: {cfg["image_max_pixels"]}
image_min_pixels: {cfg["image_min_pixels"]}

{method_block}

### dataset
dataset: {cfg["ds_s1_train"]}
template: {cfg["template"]}
cutoff_len: 8192
overwrite_cache: false
preprocessing_num_workers: 16
media_dir: ../data

### output
output_dir: {output_dir}
logging_steps: 1
save_strategy: {s1["save_strategy"]}
{save_steps_line}save_total_limit: 5
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: {cfg["per_device_train_batch_size"]}
gradient_accumulation_steps: {s1["gradient_accumulation_steps"]}
learning_rate: {s1["lr"]}
num_train_epochs: {s1["epochs"]}
lr_scheduler_type: {s1["lr_scheduler_type"]}
warmup_ratio: {s1["warmup_ratio"]}
weight_decay: {s1["weight_decay"]}
max_grad_norm: {s1["max_grad_norm"]}
{optim_line}{seed_line}bf16: true
gradient_checkpointing: true
{ds_line}ddp_timeout: 18000000
# resume_from_checkpoint: true
"""

            fpath = yaml_dir / f"{model_key}_world-model.yaml"
            with open(fpath, 'w') as f:
                f.write(STAGE1_CONFIG)
            print(f"[LF/{MODE}] {fpath.relative_to(Path(LF_ROOT))}")
            print(f"  model: {cfg['model_id']}, template: {cfg['template']}")
            print(f"  lr={s1['lr']}, warmup={s1['warmup_ratio']}, wd={s1['weight_decay']}, grad_norm={s1['max_grad_norm']}")
            if MODE == "lora":
                print(f"  lora: r={s1['lora_rank']}, alpha={s1['lora_alpha']}, dropout={s1['lora_dropout']}")
            print(f"  output_dir: {output_dir}")
            print()


#### [LlamaFactory] Stage 2 Training YAML

`LlamaFactory/examples/custom/GUI-Model-{MB,AC}/stage2_{full,lora}/{MODEL}_{base,world-model-full,world-model-lora}.yaml` 생성. world-model variant 는 Stage 1 merged 모델 (HF Hub) 을 base 로 지정하지만, `stage2_train.sh` 가 runtime 에 `model_name_or_path` 를 local `merged/{M}_stage1_${STAGE1_MODE}/epoch-${STAGE1_EPOCH}/` 경로로 sed 치환한다. LF_ROOT 기준 상대경로 (`../outputs/`) 사용. Qwen 계열 8개 모델 전체에 대해 생성된다.

- Full FT: `finetuning_type: full`, `learning_rate` 는 Full 안정화를 위해 자동으로 낮춤 (기본 1.5e-5, `CONFIGS[...]['stage2']['full_lr']` 로 조정 가능).
- LoRA: `finetuning_type: lora` + `lora_rank/alpha/target/dropout` 블록.

> **DeepSpeed config 분기 (Stage 1 과 동일 정책)**
> - `full` 모드: 모델별 `stage1_deepspeed` (기본 `ds_z3_config.json`) 그대로 사용.
> - `lora` 모드:
>   - `GPU_TYPE=RTX5090` (32GB VRAM): `examples/deepspeed/ds_z3_offload_config.json` (ZeRO-3 + CPU offload).
>   - 그 외 (`A100` / `H100`, 80GB VRAM): `examples/deepspeed/ds_z3_config.json` (표준 ZeRO-3).

In [ ]:
import os
from pathlib import Path

# Stage 2: full/lora 두 모드 x base / world-model-{full,lora} = 6 YAML / (model, dataset).
for MODE2 in ["full", "lora"]:
    for model_key, ds_configs in CONFIGS.items():
        for ds_name, cfg in ds_configs.items():
            # Stage 2 를 지원하지 않는 데이터셋 (MC) 은 skip.
            if ds_name in _STAGE1_ONLY:
                continue
            s2 = cfg["stage2"]
            mcfg = cfg["model_config"]
            yaml_dir = Path(LF_ROOT) / "examples" / "custom" / cfg["lf_subfolder"] / f"stage2_{MODE2}"
            yaml_dir.mkdir(parents=True, exist_ok=True)

            save_steps_line = f"save_steps: {s2['save_steps']}\n" if s2["save_strategy"] == "steps" else ""
            optim_line = f"optim: {s2['optim']}\n" if s2.get("optim") else ""
            seed_line = f"seed: {s2['seed']}\n" if s2.get("seed") is not None else ""
            _ds_path = mcfg.get("stage1_deepspeed")
            if _ds_path and MODE2 == "lora" and GPU_TYPE == "RTX5090":
                _ds_path = "examples/deepspeed/ds_z3_offload_config.json"
            ds_line = f"deepspeed: {_ds_path}\n" if _ds_path else ""

            if MODE2 == "full":
                method_block = (
                    "### method\n"
                    "stage: sft\n"
                    "do_train: true\n"
                    "finetuning_type: full\n"
                    f"freeze_vision_tower: {str(mcfg['freeze_vision_tower']).lower()}"
                )
                # Full FT 는 LoRA 대비 lr 을 낮춰 안정화.
                lr_value = s2.get("full_lr", 1.5e-5)
            else:
                method_block = (
                    "### method\n"
                    "stage: sft\n"
                    "do_train: true\n"
                    "finetuning_type: lora\n"
                    f"freeze_vision_tower: {str(mcfg['freeze_vision_tower']).lower()}\n"
                    f"lora_rank: {s2['lora_rank']}\n"
                    f"lora_alpha: {s2['lora_alpha']}\n"
                    "lora_target: all\n"
                    f"lora_dropout: {s2['lora_dropout']}"
                )
                lr_value = s2["lr"]

            COMMON_CONFIG = f"""\
### model
model_name_or_path: {{model_name_or_path}}
trust_remote_code: true
image_max_pixels: {cfg["image_max_pixels"]}
image_min_pixels: {cfg["image_min_pixels"]}

{method_block}

### dataset
dataset: {cfg["ds_s2_train"]}
template: {cfg["template"]}
cutoff_len: 10000
overwrite_cache: false
preprocessing_num_workers: 16
media_dir: ../data

### output
output_dir: {{output_dir}}
logging_steps: 1
save_strategy: {s2["save_strategy"]}
{save_steps_line}save_total_limit: 5
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: {cfg["per_device_train_batch_size"]}
gradient_accumulation_steps: {s2["gradient_accumulation_steps"]}
learning_rate: {lr_value}
num_train_epochs: {s2["epochs"]}
lr_scheduler_type: {s2["lr_scheduler_type"]}
warmup_ratio: {s2["warmup_ratio"]}
weight_decay: {s2["weight_decay"]}
max_grad_norm: {s2["max_grad_norm"]}
{optim_line}{seed_line}bf16: true
gradient_checkpointing: true
{ds_line}ddp_timeout: 18000000
# resume_from_checkpoint: true
"""

            VARIANTS = {
                "base": {
                    "model_name_or_path": cfg["model_id"],
                    "output_dir": cfg[f"save_s2_{MODE2}_base"],
                },
                "world-model-full": {
                    "model_name_or_path": cfg["hf_s1_model_full"],
                    "output_dir": cfg[f"save_s2_{MODE2}_world_from_full"],
                },
                "world-model-lora": {
                    "model_name_or_path": cfg["hf_s1_model_lora"],
                    "output_dir": cfg[f"save_s2_{MODE2}_world_from_lora"],
                },
            }

            print(f"=== {model_key} / {ds_name} Stage 2 ({MODE2}) YAML ===")
            print(f"  lr={lr_value}, warmup={s2['warmup_ratio']}, wd={s2['weight_decay']}, grad_norm={s2['max_grad_norm']}")
            for variant, params in VARIANTS.items():
                content = COMMON_CONFIG.format(
                    model_name_or_path=params["model_name_or_path"],
                    output_dir=params["output_dir"],
                )
                fpath = yaml_dir / f"{model_key}_{variant}.yaml"
                with open(fpath, 'w') as f:
                    f.write(content)
                print(f"[LF/{MODE2}] {fpath.relative_to(Path(LF_ROOT))}")
                print(f"  base:   {params['model_name_or_path']}")
                print(f"  output: {params['output_dir']}")
                print()


## 1. Stage 1 Data Preparation

`data/` 디렉토리의 Stage 1 (World Modeling) 데이터를 Train/Test Split 후 LLaMA-Factory 에 등록합니다.

**Task**: UI State (XML) + Action → Next UI State (XML)

**데이터 구조:**
- **Format**: ShareGPT (multimodal)

| File | AndroidControl | MonkeyCollection | MobiBench (eval-only) |
|------|----------------|-------------------|-----------------------|
| gui-model_stage1.jsonl | 71,047건 | ~100,000건 | 3,145건 (벤치마크 단일 파일) |
| gui-model_stage1_train.jsonl | 50,000 (default) | ~95,000 (95%) | — |
| gui-model_stage1_test_id.jsonl | 3,000 (default) | — | — |
| gui-model_stage1_test_ood.jsonl | 3,000 (default) | — | — |
| gui-model_stage1_test.jsonl | — | ~5,000 (5%) | — |
| Images | (jsonl 내 참조) | (jsonl 내 참조) | (벤치마크 원본) |

**Split 전략:**
- **AndroidControl** — `episodes_meta.jsonl.primary_app` 기반 **app-level ID/OOD** split
  (Stage 2 와 동일 partition 공유 → Stage 2 OOD 앱이 Stage 1 train 에서도 제외).
  기본 크기: train=50,000 / test_id=3,000 / test_ood=3,000 (`--stage1-{train,test-id,test-ood}-size`).
- **MonkeyCollection** — 메타 없음 → random split (seed=42, `--stage1-ratio` 기본 0.95).
- 실행: `python scripts/split_data.py --dataset AndroidControl` (또는 `... MonkeyCollection`).

**MobiBench (eval-only):**
- `data/MobiBench/gui-model_stage1.jsonl` 단일 파일이 **평가 세트 전체**로 사용됩니다 (ID/OOD split 없음).
- `stage1_eval.sh --eval-datasets MB` 시 dataset_info 엔트리 `GUI-Model-MB_stage1` 로 로드.


In [ ]:
import json
from pathlib import Path

LF_PATH = Path(LF_ROOT)
LF_DATA_DIR = LF_PATH / "data"

SHAREGPT_TAGS = {
    "role_tag": "from",
    "content_tag": "value",
    "user_tag": "human",
    "assistant_tag": "gpt",
    "system_tag": "system"
}

dataset_info_path = LF_DATA_DIR / "dataset_info.json"
if dataset_info_path.exists():
    with open(dataset_info_path, 'r', encoding='utf-8') as f:
        dataset_info = json.load(f)
else:
    dataset_info = {}

# Dataset registration is model-independent; iterate _DATASET_CONFIG directly
# (학습 대상 DS 만 train/test 쌍으로 등록).
_any_model = next(iter(CONFIGS))
for ds_name, cfg in CONFIGS[_any_model].items():
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 1 Data Registration ===\n")

    # AC  : train + test_id + test_ood (app-level ID/OOD split, Stage 2 partition 공유)
    # MC  : train + test (random split, 메타 없음)
    # AC2 : train + test (AC 와 동일 schema, 단일 test, ID/OOD 분리 없음)
    if ds_name in _SINGLE_TEST:
        required = [
            ("gui-model_stage1_train.jsonl", cfg["ds_s1_train"]),
            ("gui-model_stage1_test.jsonl",  cfg["ds_s1_test"]),
        ]
    else:
        required = [
            ("gui-model_stage1_train.jsonl",   cfg["ds_s1_train"]),
            ("gui-model_stage1_test_id.jsonl", cfg["ds_s1_test_id"]),
            ("gui-model_stage1_test_ood.jsonl", cfg["ds_s1_test_ood"]),
        ]
    for fname, _ds_key in required:
        fpath = DATA_PATH / fname
        if not fpath.exists():
            raise FileNotFoundError(
                f"{fpath} not found. Run split first:\n"
                f"  python scripts/split_data.py --dataset {ds_name}"
            )
        with open(fpath, 'r') as f:
            count = sum(1 for _ in f)
        print(f"[OK] {fname} ({count} entries, {fpath.stat().st_size / 1024 / 1024:.1f} MB)")

    DATASETS_TO_REGISTER = {
        ds_key: {
            "file_name": f"../../data/{ds_name}/{fname}",
            "formatting": "sharegpt",
            "columns": {"messages": "messages", "images": "images"},
            "tags": SHAREGPT_TAGS,
        }
        for fname, ds_key in required
    }

    for name, config in DATASETS_TO_REGISTER.items():
        dataset_info[name] = config
        print(f"[Registered] {name} -> {config['file_name']}")

# --- Eval-only benchmarks (MobiBench) : 단일 파일 entry ---
# MB 는 학습에 사용하지 않지만 평가 (stage1_eval.sh --eval-datasets MB) 시
# LlamaFactory dataset_info 에 등록된 이름이 필요하다. ID/OOD split 없이
# gui-model_stage1.jsonl 전체를 단일 evaluation set 으로 사용.
# NOTE: scripts/_common.sh::ensure_eval_only_dataset_info() 가 동일 엔트리를
#       평가 시작 시 자동 보장한다. 이 셀은 노트북 워크플로에서 같은 결과를
#       미리 기록해두는 역할 (idempotent — 두 경로 모두 같은 JSON 을 쓴다).
for bench_name, bcfg in _EVAL_ONLY_BENCHMARKS.items():
    bpath = Path(bcfg["data_dir"]) / bcfg["stage1_jsonl"]
    print(f"\n{'='*60}")
    print(f"=== {bench_name} Stage 1 Benchmark Registration (eval-only) ===\n")
    if not bpath.exists():
        print(f"[skip] {bpath} not found — benchmark file not staged. "
              f"Stage 1 eval on {bench_name} will fail until this file exists.")
        continue
    with open(bpath, 'r') as f:
        count = sum(1 for _ in f)
    print(f"[OK] {bcfg['stage1_jsonl']} ({count} entries, {bpath.stat().st_size / 1024 / 1024:.1f} MB)")
    dataset_info[bcfg["ds_s1_name"]] = {
        "file_name": f"../../data/{bench_name}/{bcfg['stage1_jsonl']}",
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": SHAREGPT_TAGS,
    }
    print(f"[Registered] {bcfg['ds_s1_name']} -> ../../data/{bench_name}/{bcfg['stage1_jsonl']}")

with open(dataset_info_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, indent=2, ensure_ascii=False)
print(f"\n[saved] {dataset_info_path}")


In [ ]:
import json
from pathlib import Path

_any_model = next(iter(CONFIGS))
for ds_name, cfg in CONFIGS[_any_model].items():
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 1 Dataset Statistics ===\n")

    # MC: _train/_test  AC: _train + _test_id/_test_ood (둘 중 존재하는 것만 통계)
    for fname in [
        "gui-model_stage1.jsonl",
        "gui-model_stage1_train.jsonl",
        "gui-model_stage1_test.jsonl",
        "gui-model_stage1_test_id.jsonl",
        "gui-model_stage1_test_ood.jsonl",
    ]:
        fpath = DATA_PATH / fname
        if fpath.exists():
            with open(fpath, 'r') as f:
                entries = [json.loads(line) for line in f if line.strip()]
            sample = entries[0] if entries else None
            msg_count = len(sample['messages']) if sample else 0
            img_count = len(sample.get('images', [])) if sample else 0
            print(f"  {fname}")
            print(f"   Entries: {len(entries)}")
            print(f"   Messages per sample: {msg_count}")
            print(f"   Images per sample: {img_count}")
            print(f"   File size: {fpath.stat().st_size / 1024 / 1024:.1f} MB")
            print()

    img_dir = DATA_PATH / "images"
    if img_dir.exists():
        imgs = list(img_dir.glob("*.png"))
        total_size = sum(f.stat().st_size for f in imgs) / 1024 / 1024 / 1024
        print(f"  Image directory: {img_dir}")
        print(f"   Total images: {len(imgs)}")
        print(f"   Total size: {total_size:.2f} GB")

## 2. Stage 2 Data Preparation

Stage 2 (Action Prediction) 데이터를 **app-level in-domain / out-of-domain split** 으로 분리한 뒤 LlamaFactory 에 등록합니다.

**Task**: Screenshot + UI State (XML) + Task Description → Action (JSON)

**데이터 구조:**
- **Format**: ShareGPT (multimodal)
- **Images**: Stage 1 과 동일 (공유)

**학습 대상 DS:** AndroidControl 만 (MonkeyCollection 은 Stage 2 데이터 없음 — Stage 1 전용).

**Split 워크플로 (AC):**
1. 메타데이터 추출 — `scripts/extract_androidcontrol_metadata.py` 로 GCS TFRecord 의 `actions` 중 첫 `open_app` 의 `app_name` 을 추출해 `episodes_meta.jsonl` 생성.
2. Split — `scripts/split_data.py --dataset AndroidControl` 가 앱 집합을 **in-domain / out-of-domain** 으로 나눈 뒤 각 풀에서 action-type stratified 샘플링 (largest-remainder).

| 파일 | AndroidControl (기본) |
|------|----------------------|
| `gui-model_stage2_train.jsonl` | 50,000 |
| `gui-model_stage2_test_id.jsonl` (train 에 등장한 앱) | 3,000 |
| `gui-model_stage2_test_ood.jsonl` (train 에 없는 앱) | 3,000 |

**MobiBench (eval-only, cross-benchmark):**
- `data/MobiBench/gui-model_stage2.jsonl` 단일 파일을 **single-pair overall** 로 채점합니다 (ID/OOD split 없음).
- dataset_info 엔트리: `GUI-Model-MB_stage2`. `stage2_eval.sh --eval-datasets MB` 시 `_action_eval.py score --test ... --pred ...` 경로로 `overall` 섹션 1개만 기록.

**Split 전략 (AC):**
- App 단위로 ID / OOD 분리 → 같은 앱 에피소드가 두 test set 에 섞이지 않는다.
- 각 분할 내부는 action type 비율을 원본과 맞추는 stratified subsampling.
- Random seed: 42.


In [ ]:
import json
from pathlib import Path

LF_PATH = Path(LF_ROOT)
LF_DATA_DIR = LF_PATH / "data"

SHAREGPT_TAGS = {
    "role_tag": "from",
    "content_tag": "value",
    "user_tag": "human",
    "assistant_tag": "gpt",
    "system_tag": "system"
}

dataset_info_path = LF_DATA_DIR / "dataset_info.json"
if dataset_info_path.exists():
    with open(dataset_info_path, 'r', encoding='utf-8') as f:
        dataset_info = json.load(f)
else:
    dataset_info = {}

_any_model = next(iter(CONFIGS))
for ds_name, cfg in CONFIGS[_any_model].items():
    # MC 는 Stage 2 데이터가 없으므로 skip.
    if ds_name in _STAGE1_ONLY:
        print(f"[skip] {ds_name}: Stage 1 전용 데이터셋 — Stage 2 entries 등록하지 않음.")
        continue
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 2 Data Registration ===\n")

    # AC  : train + test_id + test_ood (app-level ID/OOD split)
    # AC2 : train + test (단일 test, ID/OOD 분리 없음)
    if ds_name in _SINGLE_TEST:
        required = [
            ("gui-model_stage2_train.jsonl", cfg["ds_s2_train"]),
            ("gui-model_stage2_test.jsonl",  cfg["ds_s2_test"]),
        ]
    else:
        required = [
            ("gui-model_stage2_train.jsonl",    cfg["ds_s2_train"]),
            ("gui-model_stage2_test_id.jsonl",  cfg["ds_s2_test_id"]),
            ("gui-model_stage2_test_ood.jsonl", cfg["ds_s2_test_ood"]),
        ]
    for fname, _ds_key in required:
        fpath = DATA_PATH / fname
        if not fpath.exists():
            raise FileNotFoundError(
                f"{fpath} not found. Run split first:\n"
                f"  python scripts/extract_androidcontrol_metadata.py\n"
                f"  python scripts/split_data.py --dataset {ds_name}"
            )
        with open(fpath, 'r') as f:
            count = sum(1 for _ in f)
        print(f"[OK] {fname} ({count} entries, {fpath.stat().st_size / 1024 / 1024:.1f} MB)")

    DATASETS_TO_REGISTER = {
        ds_key: {
            "file_name": f"../../data/{ds_name}/{fname}",
            "formatting": "sharegpt",
            "columns": {"messages": "messages", "images": "images"},
            "tags": SHAREGPT_TAGS,
        }
        for fname, ds_key in required
    }

    for name, config in DATASETS_TO_REGISTER.items():
        dataset_info[name] = config
        print(f"[Registered] {name} -> {config['file_name']}")

# --- Eval-only benchmarks (MobiBench) : 단일 파일 entry (single-pair 평가) ---
# MB Stage 2 는 ID/OOD split 없이 gui-model_stage2.jsonl 전체를 overall 1-섹션
# single-pair 모드로 채점한다 (_action_eval.py score --test --pred).
# NOTE: scripts/_common.sh::ensure_eval_only_dataset_info() 가 동일 엔트리를
#       평가 시작 시 자동 보장한다. 이 셀은 노트북 워크플로에서 같은 결과를
#       미리 기록해두는 역할 (idempotent — 두 경로 모두 같은 JSON 을 쓴다).
for bench_name, bcfg in _EVAL_ONLY_BENCHMARKS.items():
    bpath = Path(bcfg["data_dir"]) / bcfg["stage2_jsonl"]
    print(f"\n{'='*60}")
    print(f"=== {bench_name} Stage 2 Benchmark Registration (eval-only) ===\n")
    if not bpath.exists():
        print(f"[skip] {bpath} not found — benchmark file not staged. "
              f"Stage 2 eval on {bench_name} will fail until this file exists.")
        continue
    with open(bpath, 'r') as f:
        count = sum(1 for _ in f)
    print(f"[OK] {bcfg['stage2_jsonl']} ({count} entries, {bpath.stat().st_size / 1024 / 1024:.1f} MB)")
    dataset_info[bcfg["ds_s2_name"]] = {
        "file_name": f"../../data/{bench_name}/{bcfg['stage2_jsonl']}",
        "formatting": "sharegpt",
        "columns": {"messages": "messages", "images": "images"},
        "tags": SHAREGPT_TAGS,
    }
    print(f"[Registered] {bcfg['ds_s2_name']} -> ../../data/{bench_name}/{bcfg['stage2_jsonl']}")

with open(dataset_info_path, 'w', encoding='utf-8') as f:
    json.dump(dataset_info, f, indent=2, ensure_ascii=False)
print(f"\n[saved] {dataset_info_path}")


In [ ]:
import json
from pathlib import Path
from collections import Counter

_any_model = next(iter(CONFIGS))
for ds_name, cfg in CONFIGS[_any_model].items():
    if ds_name in _STAGE1_ONLY:
        continue
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 2 Dataset Statistics ===\n")

    for fname in ["gui-model_stage2.jsonl", "gui-model_stage2_train.jsonl", "gui-model_stage2_test.jsonl"]:
        fpath = DATA_PATH / fname
        if fpath.exists():
            with open(fpath, 'r') as f:
                entries = [json.loads(line) for line in f if line.strip()]
            print(f"  {fname}")
            print(f"   Entries: {len(entries)}")
            print(f"   File size: {fpath.stat().st_size / 1024 / 1024:.1f} MB")

            action_types = []
            for entry in entries:
                try:
                    action = json.loads(entry['messages'][-1]['value'])
                    action_types.append(action.get('type', 'unknown'))
                except:
                    action_types.append('parse_error')

            type_counts = Counter(action_types)
            total = len(action_types)
            print(f"   Action type distribution:")
            for atype, count in type_counts.most_common():
                print(f"     {atype}: {count} ({count/total:.1%})")
            print()

In [ ]:
import json
from pathlib import Path
from collections import Counter

_any_model = next(iter(CONFIGS))
for ds_name, cfg in CONFIGS[_any_model].items():
    if ds_name in _STAGE1_ONLY:
        continue
    DATA_PATH = Path(cfg["data_dir"])
    print(f"\n{'='*60}")
    print(f"=== {ds_name} Stage 2 Action Type Analysis ===")

    train_path = DATA_PATH / "gui-model_stage2_train.jsonl"
    test_path = DATA_PATH / "gui-model_stage2_test.jsonl"

    for label, fpath in [("Train", train_path), ("Test", test_path)]:
        if not fpath.exists():
            print(f"[SKIP] {label}: {fpath.name} not found")
            continue

        with open(fpath, 'r') as f:
            entries = [json.loads(line) for line in f if line.strip()]

        action_types = []
        for entry in entries:
            try:
                action = json.loads(entry['messages'][-1]['value'])
                action_types.append(action.get('type', 'unknown'))
            except:
                action_types.append('parse_error')

        type_counts = Counter(action_types)
        total = len(action_types)
        print(f"\n  {label} Set Action Types ({total} entries)")
        for atype, count in type_counts.most_common():
            print(f"    {atype}: {count} ({count/total:.1%})")

## 3. Stage 1 SFT Training (default: full)

각 셀은 `scripts/stage1_train.sh` 를 모델·데이터셋 단위로 1회 호출한다 (`--stage1-mode full` 기본). LoRA 모드는 `--stage1-mode lora` 로 변경하거나, 전체 모델 일괄 실행은 `bash scripts/stage1_train.sh --stage1-mode lora` 처럼 `--model` / `--dataset` 를 생략한다.

`scripts/stage1_train.sh` 가 `DISABLE_VERSION_CHECK=1 FORCE_TORCHRUN=1 NNODES=1 NPROC_PER_NODE=${NPROC_PER_NODE}` prefix 와 cwd 전환을 자체 처리하므로 노트북 셀은 인자만 지정한다.


In [ ]:
## [Qwen2-VL-2B] [AndroidControl] Stage 1 Full Fine-tuning
!bash scripts/stage1_train.sh --model qwen2-vl-2b --dataset AC --stage1-mode full

In [ ]:
## [Qwen2-VL-2B] [MonkeyCollection] Stage 1 Full Fine-tuning
!bash scripts/stage1_train.sh --model qwen2-vl-2b --dataset MC --stage1-mode full

In [ ]:
## [Qwen2-VL-7B] [AndroidControl] Stage 1 Full Fine-tuning
!bash scripts/stage1_train.sh --model qwen2-vl-7b --dataset AC --stage1-mode full

In [ ]:
## [Qwen2-VL-7B] [MonkeyCollection] Stage 1 Full Fine-tuning
!bash scripts/stage1_train.sh --model qwen2-vl-7b --dataset MC --stage1-mode full

In [ ]:
## [Qwen2.5-VL-3B] [AndroidControl] Stage 1 Full Fine-tuning
!bash scripts/stage1_train.sh --model qwen2.5-vl-3b --dataset AC --stage1-mode full

In [ ]:
## [Qwen2.5-VL-3B] [MonkeyCollection] Stage 1 Full Fine-tuning
!bash scripts/stage1_train.sh --model qwen2.5-vl-3b --dataset MC --stage1-mode full

In [ ]:
## [Qwen2.5-VL-7B] [AndroidControl] Stage 1 Full Fine-tuning
!bash scripts/stage1_train.sh --model qwen2.5-vl-7b --dataset AC --stage1-mode full

In [ ]:
## [Qwen2.5-VL-7B] [MonkeyCollection] Stage 1 Full Fine-tuning
!bash scripts/stage1_train.sh --model qwen2.5-vl-7b --dataset MC --stage1-mode full

In [ ]:
## [Qwen3-VL-4B] [AndroidControl] Stage 1 Full Fine-tuning
!bash scripts/stage1_train.sh --model qwen3-vl-4b --dataset AC --stage1-mode full

In [ ]:
## [Qwen3-VL-4B] [MonkeyCollection] Stage 1 Full Fine-tuning
!bash scripts/stage1_train.sh --model qwen3-vl-4b --dataset MC --stage1-mode full

In [ ]:
## [Qwen3-VL-8B] [AndroidControl] Stage 1 Full Fine-tuning
!bash scripts/stage1_train.sh --model qwen3-vl-8b --dataset AC --stage1-mode full

In [ ]:
## [Qwen3-VL-8B] [MonkeyCollection] Stage 1 Full Fine-tuning
!bash scripts/stage1_train.sh --model qwen3-vl-8b --dataset MC --stage1-mode full

## 4. Stage 1 Merge & Upload to HuggingFace (전 epoch push)

각 셀은 `scripts/stage1_merge.sh` 를 모델·데이터셋 단위로 1회 호출한다 (`--stage1-mode full` 기본). 스크립트가 모든 `outputs/{DS}/adapters/{MODEL}_stage1_{MODE}/checkpoint-*/` 를 순회하며 epoch 별 local merge + HF Hub push (`SaFD-00/{short}-{slug}world-model-stage1-{MODE}-epoch{E}`) 를 수행한다. BEST_CHECKPOINT 개념은 제거되었고 `trainer_state.json.epoch` 으로 epoch 번호를 결정한다.

요구: `.env` 에 `HF_TOKEN` 설정.


#### [LlamaFactory] Stage 1 Merge YAML

`LlamaFactory/examples/custom/GUI-Model-{MB,AC}/stage1_merge/{MODEL}_world-model.yaml` 생성. `outputs/{DS}/adapters/{MODEL}_stage1_full/BEST_CHECKPOINT` 파일을 읽어 Hungarian F1 winner 체크포인트를 `model_name_or_path` 에 주입. 파일이 없으면 해당 조합을 `[SKIP]` 경고 후 건너뛰며, 평가 완료 후 재실행하면 건너뛴 항목도 생성. Merge 결과물은 `outputs/{DS}/merged/{MODEL}_stage1_full/` 에 저장.


In [ ]:
from pathlib import Path

_created, _skipped = 0, 0

# LF Stage 1 Merge YAML — full / lora 모드별로 생성.
# LoRA 모드는 finetuning_type: lora + adapter_name_or_path 블록을 추가,
# model_name_or_path 는 원본 base model (cfg["model_id"]) 을 참조.
for MODE in ["full", "lora"]:
    for model_key, ds_configs in CONFIGS.items():
        for ds_name, cfg in ds_configs.items():
            lf_sub = cfg["lf_subfolder"]
            yaml_dir = Path(LF_ROOT) / "examples" / "custom" / lf_sub / f"stage1_merge_{MODE}"
            yaml_dir.mkdir(parents=True, exist_ok=True)

            save_path_rel = cfg[f"save_s1_{MODE}"]
            train_dir = Path(BASE_DIR) / save_path_rel.lstrip("../")
            best_file = train_dir / "BEST_CHECKPOINT"
            if not best_file.exists():
                print(f"[SKIP] [{model_key}/{ds_name}/{MODE}] BEST_CHECKPOINT not found at {best_file}. "
                      f"Run stage1 evaluation first.")
                _skipped += 1
                continue
            ckpt_name = best_file.read_text(encoding="utf-8").strip()
            ckpt_path = f"{save_path_rel}/{ckpt_name}"
            selection = f"Hungarian F1 winner: {ckpt_name}"

            if MODE == "full":
                model_block = f"""\
### model
model_name_or_path: {ckpt_path}
trust_remote_code: true
template: {cfg["template"]}
"""
            else:
                model_block = f"""\
### model
model_name_or_path: {cfg["model_id"]}
adapter_name_or_path: {ckpt_path}
trust_remote_code: true
finetuning_type: lora
template: {cfg["template"]}
"""

            MERGE_STAGE1_CONFIG = f"""\
{model_block}
### export
export_dir: {cfg[f"out_s1_merged_{MODE}"]}
export_size: 5
export_device: cpu
export_legacy_format: false
export_hub_model_id: {cfg[f"hf_s1_model_{MODE}"]}
"""

            fpath = yaml_dir / f"{model_key}_world-model.yaml"
            with open(fpath, 'w') as f:
                f.write(MERGE_STAGE1_CONFIG)
            print(f"[Created/{MODE}] {fpath.relative_to(Path(LF_ROOT))}")
            print(f"  selection: {selection}")
            print(f"  ckpt:      {ckpt_path}")
            print(f"  export:    {cfg[f'out_s1_merged_{MODE}']}")
            print(f"  hub_id:    {cfg[f'hf_s1_model_{MODE}']}")
            print()
            _created += 1

print(f"--- Stage 1 Merge YAML: {_created} created, {_skipped} skipped ---")
if _skipped > 0:
    print("Re-run this cell after completing stage1 evaluation for the skipped models.")


In [ ]:
## [Qwen2-VL-2B] [AndroidControl] Stage 1 Merge & HuggingFace Hub Upload
!bash scripts/stage1_merge.sh --model qwen2-vl-2b --dataset AC --stage1-mode full

In [ ]:
## [Qwen2-VL-2B] [MonkeyCollection] Stage 1 Merge & HuggingFace Hub Upload
!bash scripts/stage1_merge.sh --model qwen2-vl-2b --dataset MC --stage1-mode full

In [ ]:
## [Qwen2-VL-7B] [AndroidControl] Stage 1 Merge & HuggingFace Hub Upload
!bash scripts/stage1_merge.sh --model qwen2-vl-7b --dataset AC --stage1-mode full

In [ ]:
## [Qwen2-VL-7B] [MonkeyCollection] Stage 1 Merge & HuggingFace Hub Upload
!bash scripts/stage1_merge.sh --model qwen2-vl-7b --dataset MC --stage1-mode full

In [ ]:
## [Qwen2.5-VL-3B] [AndroidControl] Stage 1 Merge & HuggingFace Hub Upload
!bash scripts/stage1_merge.sh --model qwen2.5-vl-3b --dataset AC --stage1-mode full

In [ ]:
## [Qwen2.5-VL-3B] [MonkeyCollection] Stage 1 Merge & HuggingFace Hub Upload
!bash scripts/stage1_merge.sh --model qwen2.5-vl-3b --dataset MC --stage1-mode full

In [ ]:
## [Qwen2.5-VL-7B] [AndroidControl] Stage 1 Merge & HuggingFace Hub Upload
!bash scripts/stage1_merge.sh --model qwen2.5-vl-7b --dataset AC --stage1-mode full

In [ ]:
## [Qwen2.5-VL-7B] [MonkeyCollection] Stage 1 Merge & HuggingFace Hub Upload
!bash scripts/stage1_merge.sh --model qwen2.5-vl-7b --dataset MC --stage1-mode full

In [ ]:
## [Qwen3-VL-4B] [AndroidControl] Stage 1 Merge & HuggingFace Hub Upload
!bash scripts/stage1_merge.sh --model qwen3-vl-4b --dataset AC --stage1-mode full

In [ ]:
## [Qwen3-VL-4B] [MonkeyCollection] Stage 1 Merge & HuggingFace Hub Upload
!bash scripts/stage1_merge.sh --model qwen3-vl-4b --dataset MC --stage1-mode full

In [ ]:
## [Qwen3-VL-8B] [AndroidControl] Stage 1 Merge & HuggingFace Hub Upload
!bash scripts/stage1_merge.sh --model qwen3-vl-8b --dataset AC --stage1-mode full

In [ ]:
## [Qwen3-VL-8B] [MonkeyCollection] Stage 1 Merge & HuggingFace Hub Upload
!bash scripts/stage1_merge.sh --model qwen3-vl-8b --dataset MC --stage1-mode full

### 5. Stage 1 Evaluation Pipeline

**흐름: train → eval → merge.** eval 은 HF Hub 의존 없이 로컬 per-epoch checkpoint 만 소비한다.

1. **Baseline (zero-shot)** — `{model_id}` 에 대해 `vllm_infer.py` 로 `outputs/{DS}/eval/{MODEL}/stage1_eval/base/` 생성.
2. **Checkpoint sweep** — `outputs/{DS}/adapters/{MODEL}_stage1_${MODE}/epoch-*` 을 순회.
   - `full`: `--model_name_or_path <checkpoint_dir>`
   - `lora`: `--model_name_or_path {model_id} --adapter_name_or_path <checkpoint_dir>` + `max_lora_rank=8`
   - 결과: `outputs/{DS}/eval/{MODEL}/stage1_eval/{full|lora}_world_model/checkpoint-N/(generated_predictions.jsonl|hungarian_metrics.json)`
3. **Winner 선정** — `_hungarian_eval.py select` 가 `avg_hungarian_f1` 최댓값 checkpoint 를 `outputs/{DS}/adapters/{MODEL}_stage1_${MODE}/BEST_CHECKPOINT[.json]` 에 기록. 이어지는 `stage1_merge.sh` 가 이 파일을 소비한다.

쉘 경로는 `bash scripts/stage1_eval.sh --model {MODEL} --dataset {DS} --stage1-mode {full|lora}`.

> **HF Hub sweep**: `adapters/.../checkpoint-*/` 에서 epoch 을 추출해 `SaFD-00/{short}-{slug}stage1-{mode}-world-model-epoch{E}` 로 HF repo 를 조립 → `vllm_infer.py --model_name_or_path <HF id>`. 쉘 경로: `bash scripts/stage1_eval.sh --model {MODEL} --dataset {DS} --stage1-mode {full|lora}`.


In [ ]:
import json, math, os
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt

# Stage 1 Eval-Loss report — baseline vs per-checkpoint world-model sweep.
# 최신 epoch 기준으로 trainer eval_loss 를 함께 표로 뽑아 참고용 차트 생성.

def _load_eval_loss(p: Path):
    if not p.exists():
        return None
    try:
        with open(p, "r") as f:
            return json.load(f).get("eval_loss")
    except Exception:
        return None

for model_key, ds_configs in CONFIGS.items():
    for ds_name, cfg in ds_configs.items():
        short_name = cfg["short_name"]
        ds_code    = cfg["output_prefix"].rstrip("/")
        eval_loss_dir = Path(BASE_DIR) / "outputs" / ds_code / "eval" / short_name / "stage1_eval" / "eval_loss"
        base_loss = _load_eval_loss(eval_loss_dir / "base" / "eval_results.json")
        if base_loss is None:
            print(f"[SKIP] {model_key}/{ds_name}: baseline eval_results.json 없음")
            continue

        # 두 모드 각각 per-checkpoint 집계
        mode_rows = {}
        for MODE in ("full", "lora"):
            wm_dir = eval_loss_dir / f"{MODE}_world_model"
            per_ckpt = []
            if wm_dir.is_dir():
                for cp in sorted(wm_dir.glob("epoch-*"),
                                 key=lambda p: int(p.name.split("-")[-1]) if p.name.split("-")[-1].isdigit() else -1):
                    l = _load_eval_loss(cp / "eval_results.json")
                    if l is not None:
                        per_ckpt.append((cp.name, l))
            if not per_ckpt:
                continue

            # winner: BEST_CHECKPOINT 참고 (adapter 디렉토리)
            best_file = Path(BASE_DIR) / "outputs" / ds_code / "adapters" / f"{short_name}_stage1_{MODE}" / "BEST_CHECKPOINT"
            winner_name = best_file.read_text().strip() if best_file.exists() else per_ckpt[-1][0]
            mode_rows[MODE] = {"checkpoints": per_ckpt, "winner": winner_name}

        if not mode_rows:
            print(f"[SKIP] {model_key}/{ds_name}: no world-model checkpoint eval_results 발견")
            continue

        base_ppl = math.exp(base_loss)
        print(f"\n=== {model_key} / {ds_name} Stage 1 Eval-Loss ===")
        print(f"Baseline (Zero-shot)  eval_loss={base_loss:.4f}  ppl={base_ppl:.4f}")
        for MODE, info in mode_rows.items():
            print(f"[{MODE}_world_model]")
            for name, l in info["checkpoints"]:
                mark = " <-- winner" if name == info["winner"] else ""
                print(f"  {name:<18} eval_loss={l:.4f}  ppl={math.exp(l):.4f}{mark}")

        # Winner checkpoint 를 대표로 써서 막대그래프 — baseline vs winner (모드별)
        variants = [("Base\n(Zero-shot)", base_loss)]
        for MODE, info in mode_rows.items():
            w_loss = dict(info["checkpoints"])[info["winner"]]
            variants.append((f"{MODE}\n({info['winner']})", w_loss))
        labels, losses = zip(*variants)
        colors = ["#9E9E9E"] + ["#FF5722", "#3F51B5"][:len(variants) - 1]
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        b0 = axes[0].bar(labels, losses, color=colors, width=0.5)
        axes[0].set_title("Eval Loss")
        axes[0].set_ylabel("Cross-Entropy Loss")
        for bar, val in zip(b0, losses):
            axes[0].text(bar.get_x()+bar.get_width()/2, val+max(losses)*0.02, f"{val:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
        axes[0].set_ylim(0, max(losses) * 1.3); axes[0].grid(axis="y", alpha=0.3)
        perps = [math.exp(l) for l in losses]
        b1 = axes[1].bar(labels, perps, color=colors, width=0.5)
        axes[1].set_title("Perplexity"); axes[1].set_ylabel("Perplexity = exp(loss)")
        for bar, val in zip(b1, perps):
            axes[1].text(bar.get_x()+bar.get_width()/2, val+max(perps)*0.02, f"{val:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
        axes[1].set_ylim(0, max(perps) * 1.3); axes[1].grid(axis="y", alpha=0.3)
        plt.suptitle(f"Stage 1 Eval-Loss ({model_key} / {ds_name}) — baseline vs winner", fontsize=13, fontweight="bold")
        plt.tight_layout()
        img_path = eval_loss_dir / "stage1_eval_loss_evaluation.png"
        plt.savefig(img_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"[Saved] {img_path}")

        # Markdown report
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        rlines = [f"# Stage 1 Eval-Loss Report: Base vs per-checkpoint sweep ({model_key} / {ds_name})",
                  f"\n> Generated: {now}\n",
                  "## Experiment Setup\n",
                  "| Item | Value |",
                  "|------|-------|",
                  f"| Model | {cfg['model_id']} |",
                  f"| Dataset | {ds_name} |",
                  f"| Test Dataset | {cfg['ds_s1_test']} |",
                  ""]
        rlines.append("## Baseline\n")
        rlines += ["| Metric | Base (Zero-shot) |",
                   "|--------|------------------|",
                   f"| Eval Loss | {base_loss:.4f} |",
                   f"| Perplexity | {base_ppl:.4f} |", ""]
        for MODE, info in mode_rows.items():
            rlines.append(f"## World-Model (stage1_{MODE}) — per-checkpoint\n")
            rlines += [f"| Checkpoint | Eval Loss | Perplexity | vs Base |",
                       f"|------------|-----------|------------|---------|"]
            for name, l in info["checkpoints"]:
                mark = "  **(winner)**" if name == info["winner"] else ""
                rlines.append(f"| {name}{mark} | {l:.4f} | {math.exp(l):.4f} | {(base_loss - l)/base_loss:+.1%} |")
            rlines.append("")
        report = "\n".join(rlines)
        report_path = eval_loss_dir / "evaluation_report.md"
        report_path.parent.mkdir(parents=True, exist_ok=True)
        report_path.write_text(report, encoding="utf-8")
        print(f"[Saved] {report_path}")


In [ ]:
## Stage 1 Baseline (zero-shot) — base model 을 각 평가 DS 에 대해 추론.
## 학습 DS 와 무관한 단일 baseline 이므로 --train-dataset 은 AC 로 고정 (산출 경로만 달라짐).
## 평가 DS: AC, MC (in-distribution), MB (cross-dataset 벤치마크).
##
## 산출: outputs/AC/eval/{MODEL}/stage1_eval/base/on-{EVAL_DS}/hungarian_metrics.json
!bash scripts/stage1_eval.sh --train-dataset AC --eval-datasets AC,MC,MB --variants base
## (MC 학습 모델의 baseline 도 별도 기록하려면 주석 해제)
# !bash scripts/stage1_eval.sh --train-dataset MC --eval-datasets AC,MC,MB --variants base


In [ ]:
## Stage 1 world-model variant HF Hub sweep.
##
## AC / MC 두 학습 DS 각각에서 HF 에 push 된 merged repo 를
##   SaFD-00/{short}-{slug}world-model-stage1-{full|lora}-epoch{E}
## 를 pull 해 AC / MC (in-distribution) + MB (cross-benchmark) 에서 평가.
##
## 산출: outputs/{TRAIN_DS}/eval/{MODEL}/stage1_eval/{full|lora}_world_model/epoch-{E}/on-{EVAL_DS}/hungarian_metrics.json
##
## Winner 자동 선정은 없음. 결과를 보고 Stage 2 의 --stage1-epoch 를 수동 선택.
!bash scripts/stage1_eval.sh --train-dataset AC --eval-datasets AC,MC,MB \
    --variants full_world_model,lora_world_model --epochs 1,2,3

!bash scripts/stage1_eval.sh --train-dataset MC --eval-datasets AC,MC,MB \
    --variants full_world_model,lora_world_model --epochs 1,2,3


In [ ]:
import json
import re
import math
from collections import Counter
from bs4 import BeautifulSoup
from munkres import Munkres

# ── Hungarian Metric 상수 ─────────────────────────────────────────────────
INTERACTIVE_TAGS = {"button", "input", "a", "select", "textarea"}
CONTENT_TAGS     = {"p", "img", "span"}
CLICKABLE_ATTRS  = {"clickable", "long-clickable"}

W_TAG   = 3.0   # tag 불일치 패널티 (매칭 불가)
W_TEXT  = 1.5   # text 불일치
W_INDEX = 0.2   # DOM index 거리

MATCH_THRESHOLD = 1.5   # 이 이상이면 매칭 거부
INDEX_TAU       = 2     # index_acc: 차이 ≤ τ 이면 위치 정확


# ── 요소 추출 (BeautifulSoup) ─────────────────────────────────────────────

def _collect_texts(el):
    """요소 자신 + 자식 전체에서 텍스트 토큰 수집 (중복 제거, 알파벳순 join)."""
    tokens = set()
    def add(v):
        if v:
            tokens.add(v.strip())
    add(el.get("description"))
    add(el.get("id"))
    for child in el.find_all(True):
        add(child.get("description"))
        add(child.get("id"))
        t = child.get_text(strip=True)
        if t:
            tokens.add(t)
    t = el.get_text(strip=True)
    if t:
        tokens.add(t)
    return " | ".join(sorted(tokens)) if tokens else ""


def _safe_int(v, default=-1):
    try:
        return int(v)
    except (TypeError, ValueError):
        return default


def extract_elements(xml_str):
    """XML/HTML 문자열에서 interactive 요소를 평탄화하여 추출."""
    try:
        soup = BeautifulSoup(xml_str, "xml")
    except Exception:
        soup = BeautifulSoup(xml_str, "html.parser")
    elements = []
    for el in soup.find_all(True):
        tag  = el.name
        idx  = _safe_int(el.get("index", -1))
        text = _collect_texts(el)
        is_interactive = tag in INTERACTIVE_TAGS
        is_content     = (tag in CONTENT_TAGS) and bool(text)
        is_clickable   = any(el.get(a) for a in CLICKABLE_ATTRS)
        if is_interactive or is_content or is_clickable:
            elements.append({"tag": tag, "text": text, "index": idx})
    return elements


# ── 비용 함수 ─────────────────────────────────────────────────────────────

def _text_sim(a, b):
    """Jaccard 유사도 (단어 집합 기준)."""
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    sa = set(a.lower().replace("|", "").split())
    sb = set(b.lower().replace("|", "").split())
    return len(sa & sb) / len(sa | sb)


def _match_cost(e1, e2, max_idx):
    """pred 요소 e1과 gt 요소 e2 사이의 매칭 비용."""
    if e1["tag"] != e2["tag"]:
        return W_TAG
    tc = W_TEXT  * (1.0 - _text_sim(e1["text"], e2["text"]))
    ic = W_INDEX * (abs(e1["index"] - e2["index"]) / max(max_idx, 1))
    return round(tc + ic, 5)


# ── 헝가리안 매칭 ─────────────────────────────────────────────────────────

def _hungarian_match(pred, gt):
    """헝가리안 알고리즘으로 pred-gt 간 최적 1:1 매칭."""
    n, m = len(pred), len(gt)
    if n == 0 or m == 0:
        return [], []
    max_idx = max(
        (e["index"] for e in pred + gt if e["index"] >= 0),
        default=1,
    )
    matrix = [
        [_match_cost(p, g, max_idx) for g in gt]
        for p in pred
    ]
    size   = max(n, m)
    padded = [row + [MATCH_THRESHOLD * 2] * (size - len(row)) for row in matrix]
    while len(padded) < size:
        padded.append([MATCH_THRESHOLD * 2] * size)
    indexes = Munkres().compute(padded)
    pairs = []
    for i, j in indexes:
        if i < n and j < m and matrix[i][j] < MATCH_THRESHOLD:
            pairs.append((i, j, matrix[i][j]))
    return pairs, matrix


def compute_hungarian_acc(pred_str, gt_str):
    """모델 생성 XML과 정답 XML을 비교해 hungarian 기반 평가 메트릭을 반환."""
    _zero = {
        "hungarian_ea": 0.0, "hungarian_f1": 0.0,
        "hungarian_prec": 0.0, "hungarian_rec": 0.0,
        "hungarian_text": 0.0, "hungarian_idx": 0.0,
    }
    try:
        pred_els = extract_elements(pred_str)
        gt_els   = extract_elements(gt_str)
    except Exception:
        return _zero
    if not gt_els:
        return _zero

    pairs, _ = _hungarian_match(pred_els, gt_els)
    n_pred, n_gt, n_matched = len(pred_els), len(gt_els), len(pairs)

    ea   = n_matched / max(n_pred, n_gt) if max(n_pred, n_gt) > 0 else 0.0
    prec = n_matched / n_pred             if n_pred  > 0           else 0.0
    rec  = n_matched / n_gt               if n_gt    > 0           else 0.0
    f1   = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0    else 0.0

    if pairs:
        text_sims = [_text_sim(pred_els[i]["text"], gt_els[j]["text"]) for i, j, _ in pairs]
        idx_diffs = [abs(pred_els[i]["index"] - gt_els[j]["index"]) for i, j, _ in pairs]
        text_avg  = sum(text_sims) / len(text_sims)
        idx_acc   = sum(1 for d in idx_diffs if d <= INDEX_TAU) / len(idx_diffs)
    else:
        text_avg = 0.0
        idx_acc  = 0.0

    return {
        "hungarian_ea":   round(ea, 4),
        "hungarian_f1":   round(f1, 4),
        "hungarian_prec": round(prec, 4),
        "hungarian_rec":  round(rec, 4),
        "hungarian_text": round(text_avg, 4),
        "hungarian_idx":  round(idx_acc, 4),
    }


# ── 텍스트 생성 품질 메트릭 ───────────────────────────────────────────────

def calc_bleu(reference, hypothesis, max_n=4):
    """BLEU-4 score를 계산합니다."""
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    if not hyp_tokens or not ref_tokens:
        return 0.0

    bp = min(1.0, math.exp(1 - len(ref_tokens) / len(hyp_tokens)))

    precisions = []
    for n in range(1, max_n + 1):
        ref_ngrams = Counter(tuple(ref_tokens[i:i+n]) for i in range(len(ref_tokens) - n + 1))
        hyp_ngrams = Counter(tuple(hyp_tokens[i:i+n]) for i in range(len(hyp_tokens) - n + 1))

        clipped = sum(min(count, ref_ngrams.get(ng, 0)) for ng, count in hyp_ngrams.items())
        total = sum(hyp_ngrams.values())

        if total == 0:
            precisions.append(0)
        else:
            precisions.append(clipped / total)

    if any(p == 0 for p in precisions):
        return 0.0

    log_avg = sum(math.log(p) for p in precisions) / max_n
    return bp * math.exp(log_avg)


def calc_rouge_l(reference, hypothesis):
    """ROUGE-L (F1) score를 계산합니다."""
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    if not ref_tokens or not hyp_tokens:
        return 0.0

    m, n = len(ref_tokens), len(hyp_tokens)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_tokens[i-1] == hyp_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])

    lcs_len = dp[m][n]
    precision = lcs_len / n
    recall = lcs_len / m

    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


# ── Stage 1 전체 평가 함수 ────────────────────────────────────────────────

def evaluate_stage1_predictions(test_path, pred_path):
    """Stage 1 prediction 전체 평가를 수행합니다."""
    with open(test_path, 'r') as f:
        gt_entries = [json.loads(line) for line in f if line.strip()]
    with open(pred_path, 'r') as f:
        pred_entries = [json.loads(line) for line in f if line.strip()]

    results = []
    for gt_entry, pred_entry in zip(gt_entries, pred_entries):
        gt_text = gt_entry['messages'][-1]['value']
        pred_text = pred_entry.get('predict', pred_entry.get('output', ''))

        bleu = calc_bleu(gt_text, pred_text)
        rouge_l = calc_rouge_l(gt_text, pred_text)
        exact_match = 1.0 if gt_text.strip() == pred_text.strip() else 0.0
        hungarian = compute_hungarian_acc(pred_text, gt_text)

        results.append({
            'bleu': bleu, 'rouge_l': rouge_l,
            'exact_match': exact_match, 'hungarian': hungarian,
        })

    total = len(results)
    metrics = {
        'total': total,
        'avg_bleu': sum(r['bleu'] for r in results) / total if total else 0,
        'avg_rouge_l': sum(r['rouge_l'] for r in results) / total if total else 0,
        'exact_match_rate': sum(r['exact_match'] for r in results) / total if total else 0,
        'avg_hungarian_ea':   sum(r['hungarian']['hungarian_ea'] for r in results) / total if total else 0,
        'avg_hungarian_f1':   sum(r['hungarian']['hungarian_f1'] for r in results) / total if total else 0,
        'avg_hungarian_prec': sum(r['hungarian']['hungarian_prec'] for r in results) / total if total else 0,
        'avg_hungarian_rec':  sum(r['hungarian']['hungarian_rec'] for r in results) / total if total else 0,
        'avg_hungarian_text': sum(r['hungarian']['hungarian_text'] for r in results) / total if total else 0,
        'avg_hungarian_idx':  sum(r['hungarian']['hungarian_idx'] for r in results) / total if total else 0,
    }
    return metrics, results

print("[Loaded] Stage 1 evaluation functions: calc_bleu, calc_rouge_l, compute_hungarian_acc, evaluate_stage1_predictions")

In [ ]:
import json
import math
import pandas as pd
from pathlib import Path

all_stage1_metrics = {}

for model_key, ds_configs in CONFIGS.items():
    for ds_name, cfg in ds_configs.items():
        short_name = cfg["short_name"]
        op = cfg["output_prefix"]
        s1_eval_dir = Path(BASE_DIR) / "outputs" / op.rstrip("/") / "eval" / short_name / "stage1_eval"
        hung_dir = s1_eval_dir / "hungarian_matching"
        # AC 는 ID/OOD split 이라 GT 가 두 파일. 이 셀은 in-notebook 빠른 보기용으로
        # in-domain 파일을 anchor 로 쓴다 (canonical eval 은 stage1_eval.sh + _hungarian_eval.py).
        if ds_name in _STAGE1_ONLY:
            test_path = Path(cfg["data_dir"]) / "gui-model_stage1_test.jsonl"
        else:
            test_path = Path(cfg["data_dir"]) / "gui-model_stage1_test_id.jsonl"
        train_dir = Path(BASE_DIR) / cfg["save_s1"].lstrip("../")

        MODEL_PREDS = {}
        base_pred = hung_dir / "base" / "generated_predictions.jsonl"
        if base_pred.exists():
            MODEL_PREDS["Baseline (Zero-shot)"] = base_pred
        for p in sorted(hung_dir.glob("epoch-*/generated_predictions.jsonl"),
                        key=lambda x: int(x.parent.name.split("-")[1])):
            MODEL_PREDS[p.parent.name] = p

        combo_key = f"{model_key}/{ds_name}"
        print(f"\n{'='*70}")
        print(f"=== {combo_key} Stage 1 Evaluation ({len(MODEL_PREDS)} model(s)) ===")

        base_loss_path = s1_eval_dir / "eval_loss" / "base" / "eval_results.json"
        fwm_loss_path  = s1_eval_dir / "eval_loss" / "full_world_model" / "eval_results.json"
        if base_loss_path.exists() and fwm_loss_path.exists():
            with open(base_loss_path, 'r') as f: base_loss_metrics = json.load(f)
            with open(fwm_loss_path,  'r') as f: fwm_loss_metrics  = json.load(f)
            print(f"\n  Loss-based Evaluation:")
            print(f"    Base     eval_loss: {base_loss_metrics['eval_loss']:.4f}  perplexity: {math.exp(base_loss_metrics['eval_loss']):.4f}")
            print(f"    FWM      eval_loss: {fwm_loss_metrics['eval_loss']:.4f}  perplexity: {math.exp(fwm_loss_metrics['eval_loss']):.4f}")

        ds_metrics = {}
        for model_name, pred_path in MODEL_PREDS.items():
            if not pred_path.exists():
                print(f"  [SKIP] {model_name}: {pred_path} missing")
                continue
            metrics, _ = evaluate_stage1_predictions(str(test_path), str(pred_path))
            ds_metrics[model_name] = metrics
            out_metric_path = pred_path.parent / "hungarian_metrics.json"
            with open(out_metric_path, 'w', encoding='utf-8') as f:
                json.dump(metrics, f, ensure_ascii=False, indent=2)

        all_stage1_metrics[combo_key] = ds_metrics

        if ds_metrics:
            print(f"\n  === {combo_key} Stage 1 Comparison ===")
            comparison = pd.DataFrame({
                name: {
                    'BLEU-4': f"{m['avg_bleu']:.4f}", 'ROUGE-L': f"{m['avg_rouge_l']:.4f}",
                    'Exact Match': f"{m['exact_match_rate']:.1%}", 'Hungarian EA': f"{m['avg_hungarian_ea']:.4f}",
                    'Hungarian F1': f"{m['avg_hungarian_f1']:.4f}", 'Hungarian Prec': f"{m['avg_hungarian_prec']:.4f}",
                    'Hungarian Rec': f"{m['avg_hungarian_rec']:.4f}", 'Hungarian Text': f"{m['avg_hungarian_text']:.4f}",
                    'Hungarian Idx': f"{m['avg_hungarian_idx']:.4f}",
                } for name, m in ds_metrics.items()
            })
            display(comparison)

            ckpt_metrics = {k: v for k, v in ds_metrics.items() if k.startswith("checkpoint-")}
            if ckpt_metrics:
                best_name, best_m = max(ckpt_metrics.items(),
                                        key=lambda kv: (kv[1]['avg_hungarian_f1'],
                                                        int(kv[0].split('-')[1])))
                train_dir.mkdir(parents=True, exist_ok=True)
                (train_dir / "BEST_CHECKPOINT").write_text(best_name + "\n", encoding='utf-8')
                summary = {
                    "selected": best_name,
                    "metric": "avg_hungarian_f1",
                    "metric_value": best_m['avg_hungarian_f1'],
                    "candidates": [
                        {"checkpoint": k, "avg_hungarian_f1": v['avg_hungarian_f1'],
                         "avg_bleu": v['avg_bleu'], "avg_rouge_l": v['avg_rouge_l'],
                         "exact_match_rate": v['exact_match_rate']}
                        for k, v in ckpt_metrics.items()
                    ],
                }
                (train_dir / "BEST_CHECKPOINT.json").write_text(
                    json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding='utf-8')
                print(f"\n  [{combo_key}] Hungarian F1 winner: {best_name} (F1={best_m['avg_hungarian_f1']:.4f})")
                print(f"     -> {train_dir / 'BEST_CHECKPOINT'}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

for combo_key, ds_metrics in all_stage1_metrics.items():
    if not ds_metrics:
        continue
    model_key, ds_name = combo_key.split("/")
    cfg = CONFIGS[model_key][ds_name]
    short_name = cfg["short_name"]
    model_names = list(ds_metrics.keys())
    n_models = len(model_names)
    colors = ['#9E9E9E', '#FF5722', '#2196F3', '#4CAF50'][:n_models]

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    text_metrics = ['avg_bleu', 'avg_rouge_l', 'exact_match_rate']
    text_labels = ['BLEU-4', 'ROUGE-L', 'Exact Match']
    x1 = np.arange(len(text_metrics))
    width = 0.8 / n_models

    for i, name in enumerate(model_names):
        values = [ds_metrics[name][m] for m in text_metrics]
        offset = (i - n_models / 2 + 0.5) * width
        axes[0].bar(x1 + offset, values, width, label=name, color=colors[i])

    axes[0].set_xlabel('Metric'); axes[0].set_ylabel('Score')
    axes[0].set_title('Text Generation Metrics')
    axes[0].set_xticks(x1); axes[0].set_xticklabels(text_labels, rotation=15)
    axes[0].legend(fontsize=8); axes[0].set_ylim(0, 1.0); axes[0].grid(axis='y', alpha=0.3)

    hung_metrics = ['avg_hungarian_ea', 'avg_hungarian_f1', 'avg_hungarian_prec',
                    'avg_hungarian_rec', 'avg_hungarian_text', 'avg_hungarian_idx']
    hung_labels = ['EA', 'F1', 'Precision', 'Recall', 'Text Sim', 'Index Acc']
    x2 = np.arange(len(hung_metrics))

    for i, name in enumerate(model_names):
        values = [ds_metrics[name][m] for m in hung_metrics]
        offset = (i - n_models / 2 + 0.5) * width
        axes[1].bar(x2 + offset, values, width, label=name, color=colors[i])

    axes[1].set_xlabel('Metric'); axes[1].set_ylabel('Score')
    axes[1].set_title('Hungarian Element Matching')
    axes[1].set_xticks(x2); axes[1].set_xticklabels(hung_labels, rotation=15)
    axes[1].legend(fontsize=8); axes[1].set_ylim(0, 1.0); axes[1].grid(axis='y', alpha=0.3)

    plt.suptitle(f'Stage 1 Evaluation ({combo_key})', fontsize=14, fontweight='bold')
    plt.tight_layout()

    op = cfg["output_prefix"]
    s1_eval_dir = Path(BASE_DIR) / "outputs" / op.rstrip("/") / "eval" / short_name / "stage1_eval"
    save_path = str(s1_eval_dir / "hungarian_matching" / "stage1_hungarian_matching_evaluation.png")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[Saved] {save_path}")

In [ ]:
import json, math
from pathlib import Path
from datetime import datetime

# Stage 1 Hungarian Matching report — baseline + per-checkpoint sweep.
# 사전 준비: all_stage1_metrics[f"{model_key}/{ds_name}"] 은 다음 구조여야 한다.
#   {
#     "base":                 <metrics dict>,
#     "full_world_model":     {"<ckpt>": <metrics dict>, ...},   # optional
#     "lora_world_model":     {"<ckpt>": <metrics dict>, ...},   # optional
#   }
# (Cell 57 이 per-checkpoint hungarian_metrics.json 을 이 구조로 적재하도록 갱신됨.)

def _winner_ckpt(adapter_dir: Path, fallback_key: str):
    bc = adapter_dir / "BEST_CHECKPOINT"
    if bc.exists():
        return bc.read_text().strip()
    return fallback_key

def _metric_row(label: str, m: dict, keys):
    cells = [f"{m[key]:{fmt}}" if key in m else "-" for key, _lbl, fmt in keys]
    return f"| {label} | " + " | ".join(cells) + " |"

METRIC_KEYS = [
    ("avg_bleu",          "BLEU-4",         ".4f"),
    ("avg_rouge_l",       "ROUGE-L",        ".4f"),
    ("exact_match_rate",  "Exact Match",    ".1%"),
    ("avg_hungarian_ea",  "Hungarian EA",   ".4f"),
    ("avg_hungarian_f1",  "Hungarian F1",   ".4f"),
    ("avg_hungarian_prec","Hungarian Prec", ".4f"),
    ("avg_hungarian_rec", "Hungarian Rec",  ".4f"),
    ("avg_hungarian_text","Hungarian Text", ".4f"),
    ("avg_hungarian_idx", "Hungarian Idx",  ".4f"),
]

for combo_key, entries in all_stage1_metrics.items():
    model_key, ds_name = combo_key.split("/")
    cfg = CONFIGS[model_key][ds_name]
    short_name = cfg["short_name"]
    ds_code    = cfg["output_prefix"].rstrip("/")
    s1_eval_dir = Path(BASE_DIR) / "outputs" / ds_code / "eval" / short_name / "stage1_eval"
    base_metrics = entries.get("base")

    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    rlines = [f"# Stage 1 Hungarian Matching Report ({model_key} / {ds_name})",
              f"\n> Generated: {now}\n",
              "## Experiment Setup\n",
              "| Item | Value |",
              "|------|-------|",
              f"| Model | {cfg['model_id']} |",
              f"| Dataset | {ds_name} |",
              f"| Training | {cfg['stage1']['epochs']} epochs, LR={cfg['stage1']['lr']} (cosine) |",
              ""]
    if base_metrics:
        rlines.append("## Baseline (Zero-shot)\n")
        rlines += ["| Metric | Score |", "|--------|-------|"]
        for key, lbl, fmt in METRIC_KEYS:
            if key in base_metrics:
                rlines.append(f"| {lbl} | {base_metrics[key]:{fmt}} |")
        rlines.append("")

    for MODE in ("full", "lora"):
        ckpt_map = entries.get(f"{MODE}_world_model")
        if not ckpt_map:
            continue
        adapter_dir = Path(BASE_DIR) / "outputs" / ds_code / "adapters" / f"{short_name}_stage1_{MODE}"
        winner = _winner_ckpt(adapter_dir, max(ckpt_map.keys(),
                              key=lambda c: int(c.split("-")[-1]) if c.split("-")[-1].isdigit() else -1))
        rlines.append(f"## World-Model (stage1_{MODE}) — per-checkpoint sweep\n")
        header = ["Checkpoint"] + [lbl for _, lbl, _ in METRIC_KEYS]
        rlines.append("| " + " | ".join(header) + " |")
        rlines.append("|" + "|".join(["--------"] * len(header)) + "|")
        for name, m in sorted(ckpt_map.items(),
                              key=lambda kv: int(kv[0].split("-")[-1]) if kv[0].split("-")[-1].isdigit() else -1):
            mark = "  **(winner)**" if name == winner else ""
            row = [name + mark] + [f"{m[key]:{fmt}}" if key in m else "-" for key, _, fmt in METRIC_KEYS]
            rlines.append("| " + " | ".join(row) + " |")
        rlines.append("")

    report = "\n".join(rlines)
    report_path = s1_eval_dir / "hungarian_matching" / "evaluation_report.md"
    report_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.write_text(report, encoding="utf-8")
    print(f"[Saved] {report_path}")

print("\nDone.")


## 6. Stage 2 SFT Training

**핵심 가설**: GUI World Modeling 으로 사전학습된 모델은 동일 베이스 대비 Action Prediction SFT 에서 더 높은 성능을 보인다. Stage 2 학습 DS 는 **AndroidControl (AC)** 만이며, MC 는 Stage 2 데이터가 없으므로 Stage 2 파이프라인에서 자동 skip 된다.

| ID | 역할 | 모델 | AndroidControl HF ID 패턴 |
|----|------|------|---------------------------|
| [stage2] | Base | 각 모델의 원본 | `{model_id}` (예: `Qwen/Qwen3-VL-8B-Instruct`) |
| [stage1+stage2] | World Model | Stage 1 Merged | `SaFD-00/{model}-ac-world-model-stage1-{full|lora}-epoch{E1}` |

> `{model}` = 모델 short_name, `{model_id}` = Cell 6 `_MODEL_CONFIG` 의 원본 HuggingFace ID. Stage 1 계보는 `--stage1-mode / --stage1-epoch` 로 지정.

**Base 비교:** 각 모델의 원본 베이스 (Zero-shot, 학습 없음) 도 함께 평가.

**핵심 비교:**
- stage2 vs stage1+stage2 → World Modeling 사전학습이 Action Prediction 에 미치는 효과.
- 평가 DS 는 AC (in-distribution, ID + OOD) + MB (cross-benchmark, overall only).

**공통 학습 설정 (AC):**

| 항목 | AndroidControl |
|------|----------------|
| Method | LoRA (r=32, α=64, dropout=0.1) — 7-9B 기준, 2B/3-4B 는 size tier 값 |
| Dataset | GUI-Model-AC_stage2_train (~50,000건) |
| Effective Batch | 64 (NPROC × per_device × grad_accum) |
| Eval Batch (per-device) | 4 |
| Learning Rate | 5e-5 (cosine, warmup=0.03) — size tier 로 2B/3-4B 는 6e-5 / 5e-5 |
| Epochs | 3 |
| Save/Eval Strategy | epoch |
| Weight Decay | 0.01 |
| max_grad_norm | 1.0 |
| Precision | bf16 |

> AC LoRA rank/alpha (r=16→32, α=32→64) 는 50K 샘플 규모에 맞춘 capacity 확장입니다 (vlm-gui-finetuning-research.md 권장 64-128 의 보수 선택).


### 통합 실행 (쉘 스크립트, 권장)

아래 모델별 셀 (Stage 2, AC 전용, 6개) 은 `scripts/stage2_train.sh` 를 모델당 1회 호출하여 base + world-model variant 를 한 번에 학습한다. 전체 모델을 한 번에 돌리려면 아래 통합 명령을 사용한다 (world-model variant 는 `--stage1-epoch` 가 필수).

```bash
# Stage 2 LoRA (Stage 1 full epoch 3 계보) — AC 전체 모델
!bash scripts/stage2_train.sh --dataset AC --stage1-mode full --stage1-epoch 3 --stage2-mode lora

# Stage 2 Full FT (Stage 1 full epoch 3 계보)
!bash scripts/stage2_train.sh --dataset AC --stage1-mode full --stage1-epoch 3 --stage2-mode full

# 특정 모델만
!bash scripts/stage2_train.sh --model qwen3-vl-8b --dataset AC \
    --stage1-mode full --stage1-epoch 3 --stage2-mode lora
```

> MonkeyCollection 은 Stage 2 데이터가 없어 Stage 2 학습 대상이 아닙니다. MobiBench 는 평가 전용 벤치마크이므로 `--dataset MB` 는 `scripts/_common.sh::parse_args` 에서 거절됩니다.


In [ ]:
# Stage 2 통합 학습 (shell) — 기본: Stage 1 full epoch 3 계보, Stage 2 LoRA.
# 필요에 따라 주석 해제 후 실행.
# !bash scripts/stage2_train.sh --stage1-mode full --stage1-epoch 3 --stage2-mode lora
# !bash scripts/stage2_train.sh --stage1-mode full --stage1-epoch 3 --stage2-mode full
# !bash scripts/stage2_train.sh --stage1-mode lora --stage1-epoch 3 --stage2-mode lora
# !bash scripts/stage2_train.sh --stage1-mode lora --stage1-epoch 3 --stage2-mode full
print("Use one of the stage2_train.sh invocations above (or see each model-specific cell below).")


In [ ]:
## [Qwen2-VL-2B] [AndroidControl] Stage 2 Training (base + world-model)
!bash scripts/stage2_train.sh --model qwen2-vl-2b --dataset AC \
    --stage1-mode full --stage1-epoch 3 --stage2-mode lora

In [ ]:
## [Qwen2-VL-7B] [AndroidControl] Stage 2 Training (base + world-model)
!bash scripts/stage2_train.sh --model qwen2-vl-7b --dataset AC \
    --stage1-mode full --stage1-epoch 3 --stage2-mode lora

In [ ]:
## [Qwen2.5-VL-3B] [AndroidControl] Stage 2 Training (base + world-model)
!bash scripts/stage2_train.sh --model qwen2.5-vl-3b --dataset AC \
    --stage1-mode full --stage1-epoch 3 --stage2-mode lora

In [ ]:
## [Qwen2.5-VL-7B] [AndroidControl] Stage 2 Training (base + world-model)
!bash scripts/stage2_train.sh --model qwen2.5-vl-7b --dataset AC \
    --stage1-mode full --stage1-epoch 3 --stage2-mode lora

In [ ]:
## [Qwen3-VL-4B] [AndroidControl] Stage 2 Training (base + world-model)
!bash scripts/stage2_train.sh --model qwen3-vl-4b --dataset AC \
    --stage1-mode full --stage1-epoch 3 --stage2-mode lora

In [ ]:
## [Qwen3-VL-8B] [AndroidControl] Stage 2 Training (base + world-model)
!bash scripts/stage2_train.sh --model qwen3-vl-8b --dataset AC \
    --stage1-mode full --stage1-epoch 3 --stage2-mode lora

## 7. Stage 2 Merge & Upload to HuggingFace (전 epoch push)

Stage 2 merge 는 **`scripts/stage2_merge.sh` 가 runtime 에 임시 YAML 을 만들어 처리**한다. BEST_CHECKPOINT 개념은 제거되었고, `--stage1-mode / --stage1-epoch / --stage2-mode` 플래그로 동작이 결정된다. 학습 DS 는 AC 만 (MC 는 Stage 2 데이터 없음).

**플래그:**
- `--stage1-mode {full|lora}`  : world-model variant 가 참조할 상류 Stage 1 모드
- `--stage1-epoch N`           : world-model variant 의 base 가 될 local Stage 1 epoch (world-model 전용)
- `--stage2-mode {full|lora}`  : Stage 2 학습 방식 (기본 lora)

**Merge 대상 (2 variants × 각 stage2 epoch):**

| Variant | Base Model | Adapter / Source | Export → HF Hub |
|---------|-----------|------------------|------------------|
| base        | `{model_id}` (원본)              | `outputs/AC/adapters/{MODEL}_stage2_{M2}_base/checkpoint-*` | `SaFD-00/{short}-ac-base-stage2-{M2}-epoch{E2}` |
| world-model | local `merged/{MODEL}_stage1_{M1}/epoch-{E1}/` | `outputs/AC/adapters/{MODEL}_stage2_{M2}_world_model_from_{M1}/checkpoint-*` | `SaFD-00/{short}-ac-world-model-stage1-{M1}-epoch{E1}-stage2-{M2}-epoch{E2}` |

**사용법:**
```bash
# Stage 2 LoRA, Stage 1 full-epoch3 계보
bash scripts/stage2_merge.sh --model qwen3-vl-8b --dataset AC \
    --stage1-mode full --stage1-epoch 3 --stage2-mode lora

# Stage 2 Full FT (Full checkpoint → export)
bash scripts/stage2_merge.sh --model qwen3-vl-8b --dataset AC \
    --stage1-mode full --stage1-epoch 3 --stage2-mode full
```

> 다음 셀은 기존 BEST_CHECKPOINT 기반 YAML 생성을 폐기한 placeholder 다. 아래 export 실행 셀은 `bash scripts/stage2_merge.sh` 로 대체됐다 (단일 쉘 호출).


### 통합 실행 (쉘 스크립트, 권장)

아래 모델별 셀 (Stage 2 merge, AC 전용, 6개) 은 `scripts/stage2_merge.sh` 를 모델당 1회 호출하여 base + world-model variant 의 모든 epoch 을 각각 merge + HF push 한다. merge YAML 은 노트북이 사전 생성하지 않고 스크립트가 runtime 에 임시 YAML 을 만든다.

```bash
# Stage 2 LoRA merge + HF push (Stage 1 full epoch 3 계보) — AC 전체 모델
!bash scripts/stage2_merge.sh --dataset AC --stage1-mode full --stage1-epoch 3 --stage2-mode lora

# Stage 2 Full FT merge + HF push
!bash scripts/stage2_merge.sh --dataset AC --stage1-mode full --stage1-epoch 3 --stage2-mode full

# 특정 모델만
!bash scripts/stage2_merge.sh --model qwen3-vl-8b --dataset AC \
    --stage1-mode full --stage1-epoch 3 --stage2-mode lora
```


In [ ]:
# Stage 2 통합 Merge + HF push (shell).
# 필요에 따라 주석 해제 후 실행.
# !bash scripts/stage2_merge.sh --stage1-mode full --stage1-epoch 3 --stage2-mode lora
# !bash scripts/stage2_merge.sh --stage1-mode full --stage1-epoch 3 --stage2-mode full
print("Use one of the stage2_merge.sh invocations above (or see each model-specific cell below — LEGACY).")


#### [LlamaFactory] Stage 2 Merge YAML

Stage 2 Merge YAML 은 더 이상 노트북에서 사전 생성하지 않는다. `scripts/stage2_merge.sh` 가 각 checkpoint 별로:

1. `trainer_state.json.epoch` 에서 Stage 2 epoch 을 파싱
2. variant 별 `model_name_or_path` / `adapter_name_or_path` / `export_hub_model_id` 를 채운 임시 YAML 을 `mktemp` 로 생성
4. 임시 YAML 은 실행 후 삭제

Full FT variant 는 checkpoint 자체가 완전한 모델이므로 adapter 블록 없이 `model_name_or_path: {ckpt_rel}` 만 지정한다. LoRA 는 `model_name_or_path: {base}` + `adapter_name_or_path: {ckpt_rel}` + `finetuning_type: lora` 블록.


In [ ]:
# Stage 2 Merge YAML 은 더 이상 노트북에서 사전 생성하지 않는다.
# scripts/stage2_merge.sh 가 runtime 에 adapter / base / HF repo id 를 조립해
# 임시 YAML 을 만들고 llamafactory-cli export 를 호출한다.
#
# 사용법:
#   bash scripts/stage2_merge.sh --model qwen2.5-vl-7b --dataset AC \
#       --stage1-mode full --stage1-epoch 3 --stage2-mode lora
#
# HF repo id 규칙 (_common.sh):
#   base variant       : SaFD-00/{short}-{slug}base-stage2-{mode2}-epoch{E2}
#   world-model variant: SaFD-00/{short}-{slug}world-model-stage1-{mode1}-epoch{E1}-stage2-{mode2}-epoch{E2}
print("[info] Stage 2 merge YAML generation moved to scripts/stage2_merge.sh.")


In [ ]:
## [Qwen2-VL-2B] [AndroidControl] Stage 2 Merge & HuggingFace Hub Upload (base + world-model)
!bash scripts/stage2_merge.sh --model qwen2-vl-2b --dataset AC \
    --stage1-mode full --stage1-epoch 3 --stage2-mode lora

In [ ]:
## [Qwen2-VL-7B] [AndroidControl] Stage 2 Merge & HuggingFace Hub Upload (base + world-model)
!bash scripts/stage2_merge.sh --model qwen2-vl-7b --dataset AC \
    --stage1-mode full --stage1-epoch 3 --stage2-mode lora

In [ ]:
## [Qwen2.5-VL-3B] [AndroidControl] Stage 2 Merge & HuggingFace Hub Upload (base + world-model)
!bash scripts/stage2_merge.sh --model qwen2.5-vl-3b --dataset AC \
    --stage1-mode full --stage1-epoch 3 --stage2-mode lora

In [ ]:
## [Qwen2.5-VL-7B] [AndroidControl] Stage 2 Merge & HuggingFace Hub Upload (base + world-model)
!bash scripts/stage2_merge.sh --model qwen2.5-vl-7b --dataset AC \
    --stage1-mode full --stage1-epoch 3 --stage2-mode lora

In [ ]:
## [Qwen3-VL-4B] [AndroidControl] Stage 2 Merge & HuggingFace Hub Upload (base + world-model)
!bash scripts/stage2_merge.sh --model qwen3-vl-4b --dataset AC \
    --stage1-mode full --stage1-epoch 3 --stage2-mode lora

In [ ]:
## [Qwen3-VL-8B] [AndroidControl] Stage 2 Merge & HuggingFace Hub Upload (base + world-model)
!bash scripts/stage2_merge.sh --model qwen3-vl-8b --dataset AC \
    --stage1-mode full --stage1-epoch 3 --stage2-mode lora

### 8. Stage 2 Evaluation Pipeline

**흐름: train → merge → eval.** eval 은 HF Hub 에 push 된 Stage 2 merged repo 만 pull 하며, 로컬 adapter/merged 디렉토리에 의존하지 않는다. 학습 DS (`--train-dataset`, 현재 AC 만) 와 평가 DS (`--eval-datasets`, AC / MB) 를 분리한다.

- **EVAL_DS=AC** : ID + OOD 두 test 파일을 함께 추론해 `action_metrics.json` 에 `overall / in_domain / out_of_domain` 3-section 으로 기록.
- **EVAL_DS=MB** : 단일 파일 `gui-model_stage2.jsonl` 을 single-pair 로 추론해 `action_metrics.json` 에 `overall` 1-section 만 기록.

1. **Baseline (zero-shot)** — `{model_id}` 에 대해 `vllm_infer.py` 로 각 EVAL_DS 를 돌려 `outputs/AC/eval/{MODEL}/stage2_eval/base/on-{EVAL_DS}/action_metrics.json` 에 저장.
2. **Variant × epoch sweep** — `--variants` 로 평가 대상을 선택:
   - `{full|lora}_base` : `SaFD-00/{short}-ac-base-stage2-{M2}-epoch{E2}`
   - `{full|lora}_world_model` : `SaFD-00/{short}-ac-world-model-stage1-{M1}-epoch{E1}-stage2-{M2}-epoch{E2}` (`--stage1-mode` / `--stage1-epoch` 로 계보 지정)
3. **메트릭 확인** — winner 자동 선정은 없다. 각 EVAL_DS 의 `action_metrics.json.overall.step_accuracy` 를 비교해 Stage 2 용 epoch 을 직접 결정한다.

쉘 경로:
```bash
bash scripts/stage2_eval.sh --model qwen3-vl-8b --train-dataset AC --eval-datasets AC,MB \
    --stage1-mode full --stage1-epoch 3 --stage2-mode lora \
    --variants base,lora_base,lora_world_model --epochs 1,2,3
```

Step Accuracy 정의: `ARCHITECTURE.md` §6. 테스트: `pytest tests/test_action_eval.py`.


In [ ]:
## Stage 2 Baseline — base model zero-shot 에 대해 in-distribution (AC: ID+OOD) + cross-benchmark (MB: overall) 평가.
## EVAL_DS=AC 는 _action_eval.py 가 overall/in_domain/out_of_domain 3-section 으로 기록.
## EVAL_DS=MB 는 단일 파일이므로 overall 1-section 만 기록 (single-pair 모드).
##
## 산출: outputs/AC/eval/{MODEL}/stage2_eval/base/on-{EVAL_DS}/action_metrics.json
!bash scripts/stage2_eval.sh --train-dataset AC --eval-datasets AC,MB --variants base


In [ ]:
## Stage 2 base variant HF Hub sweep — SaFD-00/...base-stage2-{M2}-epoch{E2}.
## 매 epoch 별로 in-distribution + cross-benchmark 평가.
##
## 산출: outputs/AC/eval/{MODEL}/stage2_eval/{full|lora}_base/epoch-{E2}/on-{EVAL_DS}/action_metrics.json
!bash scripts/stage2_eval.sh --train-dataset AC --eval-datasets AC,MB \
    --variants full_base,lora_base --epochs 1,2,3


In [ ]:
## Stage 2 world-model variant HF Hub sweep — Stage 1 계보 포함:
##   SaFD-00/...world-model-stage1-{M1}-epoch{E1}-stage2-{M2}-epoch{E2}
##
## --stage1-mode / --stage1-epoch 로 상류 Stage 1 merged repo 의 계보 번호를 지정한다.
## 아래 예시는 Stage 1 FULL epoch 3 계보 + Stage 2 full/lora 모두 sweep.
!bash scripts/stage2_eval.sh --train-dataset AC --eval-datasets AC,MB \
    --stage1-mode full --stage1-epoch 3 \
    --variants full_world_model,lora_world_model --epochs 1,2,3

## Stage 1 LORA 계보도 필요하면 주석 해제:
# !bash scripts/stage2_eval.sh --train-dataset AC --eval-datasets AC,MB \
#     --stage1-mode lora --stage1-epoch 3 \
#     --variants full_world_model,lora_world_model --epochs 1,2,3


In [ ]:
# NOTE: 이 셀은 scripts/_action_eval.py 와 글자 단위 동치를 유지하는 정본 복제다.
# 실제 평가는 ``python scripts/_action_eval.py score ...`` 또는 stage2_eval.sh 를
# 통해 실행한다. 이 셀은 로직 레퍼런스 / 빠른 디버깅용.

import json
import re
import sys
from collections import defaultdict

# Stage 2 Action Prediction evaluator (정본).
# scripts/_action_eval.py 와 글자 단위 동치를 유지한다.
# 메트릭: Step Accuracy (SA) — bounds 영구 부재 데이터셋용 정의.
#   correct = (parse_ok ∧ type==gt ∧ field_match(type))
# field_match: navigate_back/finish 항상 통과 · click/long_click 의 index ·
#              scroll.direction · open_app.params.app · input.params.text

# ── Action Parsing ───────────────────────────────────────────────────────
def parse_action(text):
    text = (text or "").strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except Exception:
            pass
    match = re.search(r'\{[^{}]*\}', text)
    if match:
        try:
            return json.loads(match.group())
        except Exception:
            pass
    return None


# ── Field 추출 헬퍼 (top-level + nested params 모두 지원) ────────────────
def _pval(action, key):
    if action is None:
        return None
    if key in action:
        return action[key]
    return (action.get('params') or {}).get(key)


def _norm(s):
    return str(s if s is not None else '').strip().lower()


# ── Step Accuracy 채점 ───────────────────────────────────────────────────
# field_match(type) 가 정의된 type 만 step_correct 로 인정. 그 외 type 은 False.
_FIELD_MATCH_TYPES = {
    'navigate_back', 'finish', 'click', 'long_click',
    'scroll', 'open_app', 'input',
}


def evaluate_single(gt_action, pred_action):
    result = {
        'parsed': pred_action is not None,
        'type_correct': False,
        'step_correct': False,
        'has_index_check': False,    # click / long_click
        'has_dir_check': False,      # scroll
        'has_app_check': False,      # open_app
        'has_text_check': False,     # input
    }
    if pred_action is None:
        return result

    gt_type = str(gt_action.get('type', '')).lower()
    pred_type = str(pred_action.get('type', '')).lower()
    result['type_correct'] = (gt_type == pred_type)
    if not result['type_correct']:
        return result

    # field_match
    if gt_type in ('navigate_back', 'finish'):
        result['step_correct'] = True
        return result

    if gt_type in ('click', 'long_click'):
        result['has_index_check'] = True
        result['step_correct'] = (
            str(gt_action.get('index')) == str(pred_action.get('index'))
        )
        return result

    if gt_type == 'scroll':
        result['has_dir_check'] = True
        result['step_correct'] = (
            _norm(_pval(gt_action, 'direction')) == _norm(_pval(pred_action, 'direction'))
        )
        return result

    if gt_type == 'open_app':
        result['has_app_check'] = True
        result['step_correct'] = (
            _norm(_pval(gt_action, 'app')) == _norm(_pval(pred_action, 'app'))
        )
        return result

    if gt_type == 'input':
        result['has_text_check'] = True
        result['step_correct'] = (
            _norm(_pval(gt_action, 'text')) == _norm(_pval(pred_action, 'text'))
        )
        return result

    # unknown type — type 만 일치해도 step 은 False (정책)
    return result


def evaluate_predictions(test_path, pred_path):
    with open(test_path, 'r') as f:
        gt_entries = [json.loads(line) for line in f if line.strip()]
    with open(pred_path, 'r') as f:
        pred_entries = [json.loads(line) for line in f if line.strip()]

    if len(gt_entries) != len(pred_entries):
        print(f"[warn] length mismatch: gt={len(gt_entries)} pred={len(pred_entries)} "
              f"→ truncating to {min(len(gt_entries), len(pred_entries))}", file=sys.stderr)

    results = []
    per_type = defaultdict(lambda: {
        'count': 0, 'type_correct': 0, 'step_correct': 0,
    })
    cond = {
        'index': {'n': 0, 'k': 0},   # click + long_click
        'dir':   {'n': 0, 'k': 0},   # scroll
        'app':   {'n': 0, 'k': 0},   # open_app
        'text':  {'n': 0, 'k': 0},   # input
    }

    for gt_entry, pred_entry in zip(gt_entries, pred_entries):
        gt_action = json.loads(gt_entry['messages'][-1]['value'])
        pred_text = pred_entry.get('predict', pred_entry.get('output', ''))
        pred_action = parse_action(pred_text)

        r = evaluate_single(gt_action, pred_action)
        gt_type = str(gt_action.get('type', 'unknown')).lower()
        r['gt_type'] = gt_type
        results.append(r)

        per_type[gt_type]['count'] += 1
        per_type[gt_type]['type_correct'] += int(r['type_correct'])
        per_type[gt_type]['step_correct'] += int(r['step_correct'])

        if r['has_index_check']:
            cond['index']['n'] += 1
            cond['index']['k'] += int(r['step_correct'])
        if r['has_dir_check']:
            cond['dir']['n'] += 1
            cond['dir']['k'] += int(r['step_correct'])
        if r['has_app_check']:
            cond['app']['n'] += 1
            cond['app']['k'] += int(r['step_correct'])
        if r['has_text_check']:
            cond['text']['n'] += 1
            cond['text']['k'] += int(r['step_correct'])

    total = len(results)
    parsed = sum(1 for r in results if r['parsed'])
    type_correct = sum(int(r['type_correct']) for r in results)
    step_correct = sum(int(r['step_correct']) for r in results)

    parse_rate = parsed / total if total else 0
    type_acc = type_correct / total if total else 0
    step_acc = step_correct / total if total else 0

    per_type_summary = {}
    for t, d in per_type.items():
        per_type_summary[t] = {
            'count':    d['count'],
            'type_acc': round(d['type_correct'] / d['count'] if d['count'] else 0, 4),
            'step_acc': round(d['step_correct'] / d['count'] if d['count'] else 0, 4),
        }

    macro_step = (sum(v['step_acc'] for v in per_type_summary.values()) /
                  len(per_type_summary)) if per_type_summary else 0

    def _ratio(c):
        return c['k'] / c['n'] if c['n'] else 0

    return {
        'total':                total,
        'parse_rate':           round(parse_rate, 4),
        'type_accuracy':        round(type_acc, 4),
        'step_accuracy':        round(step_acc, 4),
        'macro_step_accuracy':  round(macro_step, 4),
        'cond_index_acc':       round(_ratio(cond['index']), 4),
        'cond_dir_acc':         round(_ratio(cond['dir']),   4),
        'cond_app_acc':         round(_ratio(cond['app']),   4),
        'cond_text_acc':        round(_ratio(cond['text']),  4),
        'per_type':             per_type_summary,
    }

print("[Loaded] Step Accuracy evaluator: parse_action, evaluate_single, evaluate_predictions")


In [ ]:
## DEPRECATED (rev. 2026-04): 이 셀은 구 `gui-model_stage2_test.jsonl` 와
## legacy 메트릭 키 (avg_bounds_iou 등) 를 참조하도록 작성되어 있습니다.
## 새 파이프라인은 각 epoch 의 `action_metrics.json` 에서
##   overall.step_accuracy / in_domain.step_accuracy / out_of_domain.step_accuracy
## 를 직접 읽어 비교합니다. 요약/시각화 필요 시 본 셀을 새 키에 맞게 재작성하세요.

import json
import pandas as pd
from pathlib import Path

all_s2_metrics = {}
all_s2_results = {}

for model_key, ds_configs in CONFIGS.items():
    for ds_name, cfg in ds_configs.items():
        short_name = cfg["short_name"]
        op = cfg["output_prefix"]
        s2_eval_dir = Path(BASE_DIR) / "outputs" / op.rstrip("/") / "eval" / short_name / "stage2_eval"
        test_path = Path(cfg["data_dir"]) / "gui-model_stage2_test.jsonl"

        MODEL_PREDS = {}
        base_pred = s2_eval_dir / "base" / "generated_predictions.jsonl"
        if base_pred.exists():
            MODEL_PREDS["Base (Zero-shot)"] = base_pred
        for variant in ("lora_base", "lora_world_model"):
            for p in sorted((s2_eval_dir / variant).glob("epoch-*/generated_predictions.jsonl"),
                            key=lambda x: int(x.parent.name.split("-")[1])):
                MODEL_PREDS[f"{variant}/{p.parent.name}"] = p

        combo_key = f"{model_key}/{ds_name}"
        print(f"\n{'='*70}")
        print(f"=== {combo_key} Stage 2 Evaluation ({len(MODEL_PREDS)} model(s)) ===")

        ds_metrics = {}
        ds_results = {}
        for model_name, pred_path in MODEL_PREDS.items():
            if not pred_path.exists():
                continue
            metrics, results = evaluate_predictions(str(test_path), str(pred_path))
            ds_metrics[model_name] = metrics
            ds_results[model_name] = results
            out_metric_path = pred_path.parent / "action_metrics.json"
            with open(out_metric_path, 'w', encoding='utf-8') as f:
                json.dump({**metrics, "per_type": metrics.get("per_type", {})}, f, ensure_ascii=False, indent=2)
            print(f"[Done] {combo_key}/{model_name}  overall={metrics['overall_score']:.4f}")

        all_s2_metrics[combo_key] = ds_metrics
        all_s2_results[combo_key] = ds_results

        if ds_metrics:
            summary_df = pd.DataFrame({
                name: {
                    'Parse Rate': f"{m['parse_rate']:.1%}",
                    'Type Accuracy': f"{m['type_accuracy']:.1%}",
                    'Bounds IoU': f"{m['avg_bounds_iou']:.3f}",
                    'Params Accuracy': f"{m['params_accuracy']:.1%}",
                    'Cond. IoU': f"{m['cond_bounds_iou']:.3f}",
                    'Cond. Params': f"{m['cond_params_acc']:.1%}",
                    'Overall Score': f"{m['overall_score']:.3f}",
                } for name, m in ds_metrics.items()
            }).T
            print(f"\n=== Stage 2 Model Comparison - {combo_key} ===")
            display(summary_df)

            for variant in ("lora_base", "lora_world_model"):
                ckpt_metrics = {k.split("/")[1]: v for k, v in ds_metrics.items()
                                if k.startswith(f"{variant}/checkpoint-")}
                if not ckpt_metrics:
                    continue
                best_name, best_m = max(ckpt_metrics.items(),
                                        key=lambda kv: (kv[1]['overall_score'],
                                                        int(kv[0].split('-')[1])))
                lora_dir_rel = cfg["save_s2_base"] if variant == "lora_base" else cfg["save_s2_world"]
                lora_dir = Path(BASE_DIR) / lora_dir_rel.lstrip("../")
                lora_dir.mkdir(parents=True, exist_ok=True)
                (lora_dir / "BEST_CHECKPOINT").write_text(best_name + "\n", encoding='utf-8')
                summary = {
                    "selected": best_name,
                    "metric": "overall_score",
                    "metric_value": best_m['overall_score'],
                    "candidates": [
                        {"checkpoint": k, "overall_score": v['overall_score'],
                         "type_accuracy": v['type_accuracy'],
                         "avg_bounds_iou": v['avg_bounds_iou'],
                         "params_accuracy": v['params_accuracy'],
                         "parse_rate": v['parse_rate']}
                        for k, v in ckpt_metrics.items()
                    ],
                }
                (lora_dir / "BEST_CHECKPOINT.json").write_text(
                    json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding='utf-8')
                print(f"  [{combo_key}/{variant}] winner: {best_name} (overall={best_m['overall_score']:.4f})")
                print(f"     -> {lora_dir / 'BEST_CHECKPOINT'}")

In [ ]:
## DEPRECATED (rev. 2026-04): 이 셀은 구 `gui-model_stage2_test.jsonl` 와
## legacy 메트릭 키 (avg_bounds_iou 등) 를 참조하도록 작성되어 있습니다.
## 새 파이프라인은 각 epoch 의 `action_metrics.json` 에서
##   overall.step_accuracy / in_domain.step_accuracy / out_of_domain.step_accuracy
## 를 직접 읽어 비교합니다. 요약/시각화 필요 시 본 셀을 새 키에 맞게 재작성하세요.

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

for combo_key, ds_metrics in all_s2_metrics.items():
    if not ds_metrics:
        continue
    model_key, ds_name = combo_key.split("/")
    cfg = CONFIGS[model_key][ds_name]
    short_name = cfg["short_name"]
    model_names = list(ds_metrics.keys())
    metrics_to_plot = ['type_accuracy', 'avg_bounds_iou', 'params_accuracy', 'overall_score']
    metric_labels = ['Type Accuracy', 'Bounds IoU', 'Params Accuracy', 'Overall Score']
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    x = np.arange(len(metrics_to_plot))
    n_models = len(model_names)
    width = 0.8 / n_models
    colors = ['#9E9E9E', '#FF5722', '#2196F3', '#4CAF50'][:n_models]
    for i, name in enumerate(model_names):
        values = [ds_metrics[name][m] for m in metrics_to_plot]
        offset = (i - n_models / 2 + 0.5) * width
        axes[0].bar(x + offset, values, width, label=name, color=colors[i])
    axes[0].set_xlabel('Metric'); axes[0].set_ylabel('Score')
    axes[0].set_title(f'Stage 2: Model Comparison ({combo_key})')
    axes[0].set_xticks(x); axes[0].set_xticklabels(metric_labels, rotation=15)
    axes[0].legend(fontsize=8); axes[0].set_ylim(0, 1.0); axes[0].grid(axis='y', alpha=0.3)

    action_types = sorted(set(t for m in ds_metrics.values() for t in m['per_type'].keys()))
    x2 = np.arange(len(action_types))
    for i, name in enumerate(model_names):
        values = [ds_metrics[name]['per_type'].get(t, {}).get('type_acc', 0) for t in action_types]
        offset = (i - n_models / 2 + 0.5) * width
        axes[1].bar(x2 + offset, values, width, label=name, color=colors[i])
    axes[1].set_xlabel('Action Type'); axes[1].set_ylabel('Type Accuracy')
    axes[1].set_title(f'Per-Type Accuracy ({combo_key})')
    axes[1].set_xticks(x2); axes[1].set_xticklabels(action_types, rotation=30, fontsize=8)
    axes[1].legend(fontsize=8); axes[1].set_ylim(0, 1.0); axes[1].grid(axis='y', alpha=0.3)

    plt.tight_layout()
    op = cfg["output_prefix"]
    s2_eval_dir = Path(BASE_DIR) / "outputs" / op.rstrip("/") / "eval" / short_name / "stage2_eval"
    save_path = str(s2_eval_dir / "stage2_evaluation.png")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"[Saved] {save_path}")

In [ ]:
## DEPRECATED (rev. 2026-04): 이 셀은 구 `gui-model_stage2_test.jsonl` 와
## legacy 메트릭 키 (avg_bounds_iou 등) 를 참조하도록 작성되어 있습니다.
## 새 파이프라인은 각 epoch 의 `action_metrics.json` 에서
##   overall.step_accuracy / in_domain.step_accuracy / out_of_domain.step_accuracy
## 를 직접 읽어 비교합니다. 요약/시각화 필요 시 본 셀을 새 키에 맞게 재작성하세요.

import json
from pathlib import Path
from datetime import datetime

def generate_eval_report(model_name, metrics, results, output_dir, ds_name, cfg, comparison_metrics=None):
    m = metrics
    s2 = cfg["stage2"]
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    lines = []
    lines.append(f"# Stage 2 Evaluation Report: {model_name} ({cfg['model_key']}/{ds_name})")
    lines.append(f"\n> Generated: {now}\n")
    lines.append("## Experiment Setup\n")
    lines.append("| Item | Value |")
    lines.append("|------|-------|")
    lines.append(f"| Model | {cfg['model_id']} |")
    lines.append(f"| Dataset | {ds_name} |")
    lines.append(f"| Test Samples | {m['total']} |")
    lines.append(f"| Method | LoRA (r={s2['lora_rank']}, a={s2['lora_alpha']}, dropout={s2['lora_dropout']}) |")
    lines.append(f"| Training | {s2['epochs']} epochs, LR={s2['lr']} (cosine) |")
    lines.append("")
    lines.append("## Overall Metrics\n")
    lines.append("| Metric | Score |")
    lines.append("|--------|-------|")
    lines.append(f"| Parse Rate | {m['parse_rate']:.1%} |")
    lines.append(f"| Type Accuracy | {m['type_accuracy']:.1%} |")
    lines.append(f"| Bounds IoU (avg) | {m['avg_bounds_iou']:.4f} |")
    lines.append(f"| Bounds IoU (cond.) | {m['cond_bounds_iou']:.4f} |")
    lines.append(f"| Params Accuracy (avg) | {m['params_accuracy']:.1%} |")
    lines.append(f"| Params Accuracy (cond.) | {m['cond_params_acc']:.1%} |")
    lines.append(f"| **Overall Score** | **{m['overall_score']:.4f}** |")
    lines.append("")
    lines.append("## Per-Type Breakdown\n")
    lines.append("| Action Type | Count | Type Acc | Avg IoU | Params Acc |")
    lines.append("|-------------|-------|----------|---------|------------|")
    for t, d in sorted(m['per_type'].items(), key=lambda x: -x[1]['count']):
        lines.append(f"| {t} | {d['count']} | {d['type_acc']:.1%} | {d['avg_iou']:.4f} | {d['params_acc']:.1%} |")
    lines.append("")
    if comparison_metrics:
        lines.append("## Model Comparison\n")
        lines.append("| Metric | " + " | ".join(comparison_metrics.keys()) + " |")
        lines.append("|--------|" + "|".join(["-------"] * len(comparison_metrics)) + "|")
        for label, key, fmt in [("Parse Rate","parse_rate",".1%"),("Type Accuracy","type_accuracy",".1%"),
            ("Bounds IoU","avg_bounds_iou",".4f"),("Cond. IoU","cond_bounds_iou",".4f"),
            ("Params Accuracy","params_accuracy",".1%"),("Cond. Params","cond_params_acc",".1%"),
            ("Overall Score","overall_score",".4f")]:
            values = []
            scores = [cm[key] for cm in comparison_metrics.values()]
            best = max(scores)
            for name, cm in comparison_metrics.items():
                val = cm[key]
                formatted = f"{val:{fmt}}"
                if val == best and len(set(scores)) > 1:
                    formatted = f"**{formatted}**"
                values.append(formatted)
            lines.append(f"| {label} | " + " | ".join(values) + " |")
        lines.append("")
    report = "\n".join(lines)
    output_path = Path(output_dir) / "evaluation_report.md"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(report)
    return str(output_path)

for combo_key, ds_metrics in all_s2_metrics.items():
    model_key, ds_name = combo_key.split("/")
    cfg = CONFIGS[model_key][ds_name]
    short_name = cfg["short_name"]
    op = cfg["output_prefix"]
    s2_eval_dir = Path(BASE_DIR) / "outputs" / op.rstrip("/") / "eval" / short_name / "stage2_eval"
    OUTPUT_DIRS = {
        "Base (Zero-shot)": s2_eval_dir / "base",
        "stage2": s2_eval_dir / "lora_base",
        "stage1+stage2": s2_eval_dir / "lora_world_model",
    }
    for model_name, m in ds_metrics.items():
        output_dir = OUTPUT_DIRS.get(model_name)
        if output_dir:
            saved = generate_eval_report(model_name, m, all_s2_results[combo_key][model_name],
                                          output_dir, ds_name, cfg, comparison_metrics=ds_metrics)
            print(f"[Saved] {saved}")
print("\nDone.")